# Research notebook archive

Original project notebook with outputs and personal metadata removed. Existing Colab paths and experiment settings require configuration; see the repository README before running.

# Behavioral spillover test: base vs demographic fine-tuned model

This notebook tests whether demographic fine-tuning spills over into behavioral validity.
It uses two ground truths:
1) ANES 2024 respondent-level survey behavior: party ID, ideology, and vote-choice proxies.
2) MEDSL 2024 state presidential returns: aggregate state two-party Democratic vote share.

Edit the USER CONFIG cell before running.



In [ ]:
# Colab install cell. Restart runtime after the first install if Colab asks.
!pip install -q vllm transformers peft accelerate pyarrow pandas numpy tqdm matplotlib requests
!pip uninstall -y torchao

In [ ]:
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')



In [ ]:
# =====================================================================
# USER CONFIG: change only this block first
# =====================================================================

from pathlib import Path
import json

# Root folder where outputs and downloaded validation data will be saved.
PROJECT_ROOT = Path('/content/drive/MyDrive/NLP_project')
DATA_DIR = PROJECT_ROOT / 'data'
OUT_ROOT = DATA_DIR / 'behavioral_spillover'
RAW_DATA_DIR = DATA_DIR / 'external_validation'

# ---------------------------------------------------------------------
# Model identifiers or local paths
# ---------------------------------------------------------------------
# Your adapter_config.json says this LoRA was trained from Qwen/Qwen2.5-14B-Instruct.
BASE_MODEL_PATH = 'Qwen/Qwen2.5-14B-Instruct'

# This is a LoRA/PEFT adapter folder, not a merged full model folder.
FT_MODEL_PATH = (
    '/content/drive/MyDrive/NLP_project/data/stage3_outputs/'
    'sequential_experiments/balanced_canonical/checkpoints/step_07_after_ENGLISH_STATUS'
)

# Friendly names used in output filenames.
BASE_RUN_NAME = 'qwen25_14b_base'
FT_RUN_NAME = 'qwen25_14b_demog_lora_step07'

# Existing ACS/IPUMS cleaned data. Not used while RUN_STATE_SIMULATION=False.
ACS_CLEAN_PATH = DATA_DIR / 'demographic_data_clean.parquet'
RAW_ACS_PATH_CANDIDATES = [
    DATA_DIR / 'usa_00009.csv.gz',
    Path('/content/drive/MyDrive/Aa_cours/PhD6/6_8610/NLP_project/data/usa_00009.csv.gz'),
]

# Online validation sources.
ANES_CSV_ZIP_URL = 'https://electionstudies.org/anes_timeseries_2024_csv_20250808/'
MEDSL_STATE_RESULTS_URL = 'https://raw.githubusercontent.com/MEDSL/2024-elections-official/main/2024-president-state.csv'

# ---------------------------------------------------------------------
# Test-run experiment size
# ---------------------------------------------------------------------
RANDOM_SEED = 42

# Test run size. Increase later to 500 or 1200 for the real run.
N_ANES_PER_TASK = 1200

# Keep state simulation OFF for this slim ANES test.
N_ACS_PER_STATE = 150
RUN_ANES_MICRODATA = True
RUN_STATE_SIMULATION = True

# Runtime knobs.
BATCH_SIZE = 32
CHECKPOINT_EVERY_BATCHES = 5
DTYPE = 'auto'
GPU_MEM_UTIL = 0.85
TENSOR_PARALLEL = 1
MAX_MODEL_LEN = 4096
ENFORCE_EAGER = True
PRIMING_SUFFIX = ''

# ---------------------------------------------------------------------
# Prompt controls
# ---------------------------------------------------------------------
# full = state + demographics
# no_state = demographics only
#
# This gives the cleaner spillover comparison:
#   full: practical silicon-subject setting
#   no_state: demographics-only behavioral spillover, less state-prior contamination
PROMPT_CONDITIONS = ['full', 'no_state']
MAIN_CONDITION = 'full'

# ---------------------------------------------------------------------
# Output directory
# ---------------------------------------------------------------------
# This run tag prevents overwriting/reusing your previous smoke-test outputs.
RUN_TAG = (
    f'anes_real_n{N_ANES_PER_TASK}'
    f'_state_real_n{N_ACS_PER_STATE}_per_state'
    f'_weighted_sample_unit_eval'
    f'_tasks_vote_ideology_conditions_{"-".join(PROMPT_CONDITIONS)}'
)
RUN_ID = f'{BASE_RUN_NAME}__vs__{FT_RUN_NAME}__seed{RANDOM_SEED}__{RUN_TAG}'
OUT_DIR = OUT_ROOT / RUN_ID

for p in [
    DATA_DIR,
    OUT_ROOT,
    RAW_DATA_DIR,
    OUT_DIR,
    OUT_DIR / 'plots',
    OUT_DIR / 'scores',
    OUT_DIR / 'tables',
    OUT_DIR / 'progress',
]:
    p.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('OUT_DIR:', OUT_DIR)
print('BASE_MODEL_PATH:', BASE_MODEL_PATH)
print('BASE_MODEL_PATH is local path:', Path(BASE_MODEL_PATH).exists())
print('FT_MODEL_PATH:', FT_MODEL_PATH)
print('FT_MODEL_PATH exists:', Path(FT_MODEL_PATH).exists())
print('ACS_CLEAN_PATH:', ACS_CLEAN_PATH, ACS_CLEAN_PATH.exists())
print('N_ANES_PER_TASK:', N_ANES_PER_TASK)
print('N_ACS_PER_STATE:', N_ACS_PER_STATE)
print('RUN_ANES_MICRODATA:', RUN_ANES_MICRODATA)
print('RUN_STATE_SIMULATION:', RUN_STATE_SIMULATION)
print('PROMPT_CONDITIONS:', PROMPT_CONDITIONS)

# ---------------------------------------------------------------------
# Validate FT path
# ---------------------------------------------------------------------
ft_path = Path(FT_MODEL_PATH)

if not ft_path.exists():
    raise FileNotFoundError(
        f'FT_MODEL_PATH does not exist:\n{FT_MODEL_PATH}\n\n'
        'Check the path to the adapter checkpoint folder.'
    )

is_lora_adapter = (ft_path / 'adapter_config.json').exists() and (
    (ft_path / 'adapter_model.safetensors').exists() or
    (ft_path / 'adapter_model.bin').exists()
)

is_merged_model = (ft_path / 'config.json').exists() and (
    bool(list(ft_path.glob('*.safetensors'))) or
    bool(list(ft_path.glob('model*.safetensors'))) or
    bool(list(ft_path.glob('pytorch_model*.bin')))
)

print('FT path type:')
print('  LoRA/PEFT adapter:', is_lora_adapter)
print('  merged/full model:', is_merged_model)

if not (is_lora_adapter or is_merged_model):
    raise FileNotFoundError(
        f'FT_MODEL_PATH is neither a LoRA adapter folder nor a merged model folder:\n{FT_MODEL_PATH}'
    )

if is_lora_adapter:
    adapter_cfg = json.loads((ft_path / 'adapter_config.json').read_text())
    adapter_base = adapter_cfg.get('base_model_name_or_path')
    print('Adapter base_model_name_or_path:', adapter_base)

    if adapter_base != BASE_MODEL_PATH:
        raise ValueError(
            'Adapter base model does not match BASE_MODEL_PATH.\n'
            f'  adapter base: {adapter_base}\n'
            f'  BASE_MODEL_PATH: {BASE_MODEL_PATH}\n\n'
            'Set BASE_MODEL_PATH to the adapter base model before comparing.'
        )

    print('LoRA adapter path looks valid.')
else:
    print('Merged/full FT model path looks valid.')

In [ ]:
# =====================================================================
# Diagnose fine-tuned model path
# =====================================================================

from pathlib import Path

SEARCH_ROOTS = [
    Path("/content/drive/MyDrive/NLP_project"),
    Path("/content/drive/MyDrive/NLP_Project"),
    Path("/content/drive/MyDrive/Aa_cours/PhD6/6_8610/NLP_project"),
]

print("Searching for model folders under:")
for r in SEARCH_ROOTS:
    print(" ", r, "exists=", r.exists())

print("\nLoadable merged/base model folders: directories containing config.json")
merged_candidates = []
for root in SEARCH_ROOTS:
    if root.exists():
        merged_candidates.extend(list(root.rglob("config.json")))

merged_model_dirs = sorted({p.parent for p in merged_candidates})
for i, p in enumerate(merged_model_dirs[:80]):
    has_tok = (p / "tokenizer.json").exists() or (p / "tokenizer_config.json").exists()
    has_weights = (
        bool(list(p.glob("*.safetensors"))) or
        bool(list(p.glob("pytorch_model*.bin"))) or
        bool(list(p.glob("model*.safetensors")))
    )
    print(f"[{i}] {p}")
    print(f"    tokenizer={has_tok}, weights={has_weights}")

print("\nLoRA adapter folders: directories containing adapter_config.json")
adapter_candidates = []
for root in SEARCH_ROOTS:
    if root.exists():
        adapter_candidates.extend(list(root.rglob("adapter_config.json")))

adapter_dirs = sorted({p.parent for p in adapter_candidates})
for i, p in enumerate(adapter_dirs[:80]):
    has_adapter_weights = (
        (p / "adapter_model.safetensors").exists() or
        (p / "adapter_model.bin").exists()
    )
    print(f"[{i}] {p}")
    print(f"    adapter_weights={has_adapter_weights}")

print("\nRecommended action:")
print("1. If you see a folder ending in models/final_merged_model with config.json and weights=True, use that as FT_MODEL_PATH.")
print("2. If you only see final_adapter_unmerged, then you do not have a merged model and need adapter loading/merging.")
print("3. Make sure the FT model base matches BASE_MODEL_PATH. Do not compare Qwen2.5-14B base to a Qwen3-4B fine-tune.")

In [ ]:
# =====================================================================
# Imports and general helpers
# =====================================================================
# IMPORTANT for Colab/Jupyter:
# vLLM's V1 engine can fail with:
#   io.UnsupportedOperation: fileno
# because ipykernel stdout does not expose a normal file descriptor.
# Set these BEFORE importing vllm anywhere.
import os
os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
import io
import sys
import json
import math
import re
import string
import zipfile
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 100)
pd.set_option("display.precision", 4)
plt.rcParams["figure.dpi"] = 120


def patch_ipykernel_fileno():
    """
    Patch Colab/Jupyter stdout/stderr so libraries that call fileno()
    do not crash. This is a defensive patch for vLLM in notebooks.
    """
    for stream, fd in [(sys.stdout, 1), (sys.stderr, 2)]:
        try:
            stream.fileno()
        except Exception:
            try:
                stream.fileno = lambda fd=fd: fd
            except Exception:
                pass


patch_ipykernel_fileno()


def cleanup_memory():
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass


def safe_name(x: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(x)).strip("_")


def weighted_mean(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    mask = np.isfinite(x) & np.isfinite(w) & (w > 0)
    if mask.sum() == 0:
        return np.nan
    return float(np.sum(x[mask] * w[mask]) / np.sum(w[mask]))


def softmax(logits):
    x = np.asarray(logits, dtype=float)
    x = x - np.nanmax(x)
    e = np.exp(x)
    return e / np.sum(e)


def download_file(url: str, dest: Path, force: bool = False, timeout: int = 120) -> Path:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and not force and dest.stat().st_size > 0:
        print(f"Using cached {dest}")
        return dest
    print(f"Downloading {url}\n  -> {dest}")
    r = requests.get(url, allow_redirects=True, timeout=timeout)
    r.raise_for_status()
    dest.write_bytes(r.content)
    print(f"Saved {dest} ({dest.stat().st_size / 1e6:.2f} MB)")
    return dest


def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)


save_json({
    "BASE_MODEL_PATH": BASE_MODEL_PATH,
    "FT_MODEL_PATH": FT_MODEL_PATH,
    "BASE_RUN_NAME": BASE_RUN_NAME,
    "FT_RUN_NAME": FT_RUN_NAME,
    "ANES_CSV_ZIP_URL": ANES_CSV_ZIP_URL,
    "MEDSL_STATE_RESULTS_URL": MEDSL_STATE_RESULTS_URL,
    "N_ANES_PER_TASK": N_ANES_PER_TASK,
    "N_ACS_PER_STATE": N_ACS_PER_STATE,
    "PROMPT_CONDITIONS": PROMPT_CONDITIONS,
    "MAIN_CONDITION": MAIN_CONDITION,
    "VLLM_USE_V1": os.environ.get("VLLM_USE_V1"),
    "VLLM_WORKER_MULTIPROC_METHOD": os.environ.get("VLLM_WORKER_MULTIPROC_METHOD"),
}, OUT_DIR / "run_config.json")

print("vLLM env:")
print("  VLLM_USE_V1 =", os.environ.get("VLLM_USE_V1"))
print("  VLLM_WORKER_MULTIPROC_METHOD =", os.environ.get("VLLM_WORKER_MULTIPROC_METHOD"))

In [ ]:
# =====================================================================
# Shared prompt metadata, adapted from your Stage 1 / feature ablation notebooks
# =====================================================================
FEATURE_LABELS = {
    'SEX': 'Sex',
    'AGE_BINS': 'Age group',
    'RACE_ETHNICITY': 'Race/ethnicity',
    'EDUCATION_LEVEL': 'Highest level of education completed',
    'INCTOT_BINS': 'Total annual personal income',
    'HEALTH_INSURANCE': 'Health insurance coverage',
    'ENGLISH_STATUS': 'English speaking ability',
    'NCHILD': 'Number of own children living in household',
    'UHRSWORK': 'Usual hours worked per week',
    'STATE_NAME': 'State of residence',
    'OWNERSHP_STATUS': 'Home ownership status',
    'MARITAL_STATUS': 'Marital status',
    'EMPLOYMENT_STATUS': 'Employment status',
    'WORKER_CLASS': 'Class of worker',
    'TRANSPORT_METHOD': 'Means of transportation to work',
}

LABEL_TO_LONG = {
    'White': 'White',
    'Hispanic': 'Hispanic',
    'Black': 'Black or African American',
    'Indigenous': 'American Indian or Alaska Native',
    'Chinese': 'Chinese',
    'Japanese': 'Japanese',
    'Other': 'Other Asian or Pacific Islander',
    'Asian/PI': 'Asian or Native Hawaiian/other Pacific Islander',
    'Multiple': 'Multiple races',
    '18–24': '18 to 24 years old',
    '25–34': '25 to 34 years old',
    '35–44': '35 to 44 years old',
    '45–54': '45 to 54 years old',
    '55–64': '55 to 64 years old',
    '65+': '65 years or older',
    'Male': 'Male',
    'Female': 'Female',
}

BEHAVIOR_SYSTEM_PROMPT = '''You are simulating a U.S. survey respondent.

You will be given a demographic profile and a survey question. Predict the response this respondent would be most likely to give.

Rules:
- Use only the provided profile.
- Do not use knowledge of any specific named real election result.
- Choose exactly one option from the list.
- Reply with only the letter of the selected option — no parentheses, no explanation, no extra text.'''


def display_value(v):
    return LABEL_TO_LONG.get(v, v)


def build_behavior_profile_lines(row: pd.Series, condition: str = 'full') -> List[str]:
    if condition == 'state_only':
        cols = ['STATE_NAME']
    elif condition == 'no_state':
        cols = [c for c in FEATURE_LABELS if c != 'STATE_NAME']
    elif condition == 'full':
        cols = list(FEATURE_LABELS.keys())
    else:
        raise ValueError(f'Unknown condition: {condition}')

    lines = []
    for col in cols:
        if col not in row.index:
            continue
        value = row[col]
        if pd.isna(value):
            continue
        lines.append(f'- {FEATURE_LABELS[col]}: {display_value(value)}')
    return lines


def make_lettered_options(classes: List[str], seed: int) -> List[Tuple[str, str]]:
    rng = np.random.default_rng(int(seed))
    shuffled = list(classes)
    rng.shuffle(shuffled)
    letters = list(string.ascii_uppercase[:len(shuffled)])
    return list(zip(letters, shuffled))


def build_behavior_prompt(row: pd.Series, task: dict, condition: str, lettered_options: List[Tuple[str, str]]) -> str:
    profile_text = '\n'.join(build_behavior_profile_lines(row, condition=condition))
    answers_text = '\n'.join(f'({letter}) {label}' for letter, label in lettered_options)
    return (
        f'Profile:\n\n{profile_text}\n\n'
        f'Survey question:\n{task["question"]}\n\n'
        f'Answer options:\n{answers_text}\n\n'
        f'Reply with only the letter of the best answer.'
    )



In [ ]:
# # =====================================================================
# # vLLM letter scorer, same scoring idea as your Stage 1 notebooks
# # =====================================================================

# PROMPT_LOGPROBS = 20  # vLLM max allowed in this Colab/runtime


# class VLLMLetterScorer:
#     def __init__(self, model_name: str):
#         # Re-assert these immediately before importing vLLM.
#         os.environ["VLLM_USE_V1"] = "0"
#         os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
#         os.environ["TOKENIZERS_PARALLELISM"] = "false"
#         patch_ipykernel_fileno()
#         cleanup_memory()

#         from vllm import LLM
#         from transformers import AutoTokenizer

#         self.model_name = model_name
#         print(f"Initializing tokenizer for: {model_name}")
#         self.tokenizer = AutoTokenizer.from_pretrained(model_name)

#         if self.tokenizer.pad_token is None:
#             self.tokenizer.pad_token = self.tokenizer.eos_token

#         print(f"Initializing vLLM for: {model_name}")
#         print("vLLM env:")
#         print("  VLLM_USE_V1 =", os.environ.get("VLLM_USE_V1"))
#         print("  VLLM_WORKER_MULTIPROC_METHOD =", os.environ.get("VLLM_WORKER_MULTIPROC_METHOD"))
#         print("  PROMPT_LOGPROBS =", PROMPT_LOGPROBS)

#         self.llm = LLM(
#             model=model_name,
#             dtype=DTYPE,
#             tensor_parallel_size=TENSOR_PARALLEL,
#             gpu_memory_utilization=GPU_MEM_UTIL,
#             max_model_len=MAX_MODEL_LEN,
#             enforce_eager=ENFORCE_EAGER,
#             disable_log_stats=True,
#             trust_remote_code=False,
#         )

#     def close(self):
#         try:
#             del self.llm
#         except Exception:
#             pass
#         cleanup_memory()

#     def _build_prefix(self, system_prompt: str, user_prompt: str) -> str:
#         messages = [
#             {"role": "system", "content": system_prompt},
#             {"role": "user", "content": user_prompt},
#         ]
#         return self.tokenizer.apply_chat_template(
#             messages,
#             tokenize=False,
#             add_generation_prompt=True,
#         )

#     def _single_token_id(self, letter: str) -> int:
#         ids = self.tokenizer(letter, add_special_tokens=False).input_ids
#         if len(ids) != 1:
#             raise AssertionError(
#                 f"Letter {letter!r} tokenizes to {len(ids)} tokens, expected 1. "
#                 "Try letters A/B/C or adjust PRIMING_SUFFIX."
#             )
#         return ids[0]

#     def score_letters_batched(
#         self,
#         prompts: List[Tuple[str, str]],
#         letters_per_prompt: List[List[str]],
#     ) -> List[Dict[str, float]]:
#         patch_ipykernel_fileno()
#         from vllm import SamplingParams

#         full_prompts = []
#         index = []

#         for i, ((sys_p, user_p), letters) in enumerate(zip(prompts, letters_per_prompt)):
#             prefix = self._build_prefix(sys_p, user_p) + PRIMING_SUFFIX
#             for letter in letters:
#                 full_prompts.append(prefix + letter)
#                 index.append((i, letter))

#         params = SamplingParams(
#             temperature=0.0,
#             max_tokens=1,
#             prompt_logprobs=PROMPT_LOGPROBS,
#         )

#         outputs = self.llm.generate(full_prompts, params, use_tqdm=False)

#         results = [{} for _ in prompts]

#         for out_idx, (prompt_idx, letter) in enumerate(index):
#             out = outputs[out_idx]
#             letter_id = self._single_token_id(letter)
#             last_pos = len(out.prompt_token_ids) - 1
#             token_lps = out.prompt_logprobs[last_pos]

#             if token_lps is not None and letter_id in token_lps:
#                 lp = token_lps[letter_id].logprob
#             else:
#                 # If the letter is not in the top PROMPT_LOGPROBS tokens, treat it as very unlikely.
#                 # This should be rare for A/B/C/D after the priming suffix.
#                 lp = -100.0

#             results[prompt_idx][letter] = float(lp)

#         return results

In [ ]:
# =====================================================================
# Transformers letter scorer with PEFT/LoRA adapter support
# =====================================================================
# Drop-in replacement for the vLLM scorer.
#
# It supports:
#   1. Hugging Face base model IDs, e.g. Qwen/Qwen2.5-14B-Instruct
#   2. Full merged local model folders
#   3. LoRA/PEFT adapter folders with adapter_config.json + adapter_model.safetensors
#
# The downstream code can stay the same because the class is still named
# VLLMLetterScorer.

import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

HF_SCORE_BATCH_SIZE = 4  # lower to 1-2 if OOM; increase to 8 if GPU memory is comfortable


class VLLMLetterScorer:
    def __init__(self, model_name: str):
        cleanup_memory()

        self.model_name = model_name
        model_path = Path(model_name)
        is_local_adapter = model_path.exists() and (model_path / "adapter_config.json").exists()

        dtype = torch.bfloat16
        if not torch.cuda.is_available():
            dtype = torch.float32

        # -----------------------------------------------------------------
        # Tokenizer
        # -----------------------------------------------------------------
        if is_local_adapter and (
            (model_path / "tokenizer.json").exists()
            or (model_path / "tokenizer_config.json").exists()
        ):
            tokenizer_source = model_name
        elif is_local_adapter:
            tokenizer_source = BASE_MODEL_PATH
        else:
            tokenizer_source = model_name

        print(f"Initializing tokenizer from: {tokenizer_source}")
        self.tokenizer = AutoTokenizer.from_pretrained(
            tokenizer_source,
            trust_remote_code=False,
        )

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # Left padding is usually safer for decoder-only batched inference.
        self.tokenizer.padding_side = "left"

        # -----------------------------------------------------------------
        # Model
        # -----------------------------------------------------------------
        print(f"Initializing model for: {model_name}")
        print("  dtype:", dtype)
        print("  device_map: auto")
        print("  HF_SCORE_BATCH_SIZE:", HF_SCORE_BATCH_SIZE)

        if is_local_adapter:
            from peft import PeftModel

            adapter_cfg = json.loads((model_path / "adapter_config.json").read_text())
            adapter_base = adapter_cfg.get("base_model_name_or_path", BASE_MODEL_PATH)

            print("Detected LoRA/PEFT adapter.")
            print("  base model:", adapter_base)
            print("  adapter:", model_name)

            base_model = AutoModelForCausalLM.from_pretrained(
                adapter_base,
                torch_dtype=dtype,
                device_map="auto",
                trust_remote_code=False,
                low_cpu_mem_usage=True,
            )

            self.model = PeftModel.from_pretrained(
                base_model,
                model_name,
                torch_dtype=dtype,
                device_map="auto",
            )
        else:
            print("Detected Hugging Face model ID or merged/full local model.")
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=dtype,
                device_map="auto",
                trust_remote_code=False,
                low_cpu_mem_usage=True,
            )

        self.model.eval()

        # Precompute token ids for common answer letters.
        self._letter_token_cache = {}

    def close(self):
        try:
            del self.model
        except Exception:
            pass
        cleanup_memory()

    def _build_prefix(self, system_prompt: str, user_prompt: str) -> str:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
        return self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    def _single_token_id(self, letter: str) -> int:
        if letter in self._letter_token_cache:
            return self._letter_token_cache[letter]

        ids = self.tokenizer(letter, add_special_tokens=False).input_ids

        if len(ids) != 1:
            # Try with a leading space if the bare letter is not single-token.
            ids2 = self.tokenizer(" " + letter, add_special_tokens=False).input_ids
            if len(ids2) == 1:
                ids = ids2
            else:
                raise AssertionError(
                    f"Letter {letter!r} tokenizes to {len(ids)} tokens as bare letter "
                    f"and {len(ids2)} tokens with leading space. Expected one token."
                )

        self._letter_token_cache[letter] = ids[0]
        return ids[0]

    @torch.no_grad()
    def score_letters_batched(
        self,
        prompts: List[Tuple[str, str]],
        letters_per_prompt: List[List[str]],
    ) -> List[Dict[str, float]]:
        """
        For each prompt, compute log P(letter | chat_prompt + PRIMING_SUFFIX).

        Returns:
          List of dictionaries, one per prompt, e.g.
          [{"A": -0.2, "B": -2.1, "C": -4.8}, ...]
        """
        results = []

        full_prefixes = [
            self._build_prefix(sys_p, user_p) + PRIMING_SUFFIX
            for sys_p, user_p in prompts
        ]

        for start in range(0, len(full_prefixes), HF_SCORE_BATCH_SIZE):
            end = min(start + HF_SCORE_BATCH_SIZE, len(full_prefixes))
            batch_texts = full_prefixes[start:end]
            batch_letters = letters_per_prompt[start:end]

            enc = self.tokenizer(
                batch_texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_MODEL_LEN,
            )

            # Move tensors to the first model device.
            first_device = next(self.model.parameters()).device
            enc = {k: v.to(first_device) for k, v in enc.items()}

            out = self.model(**enc)
            logits = out.logits  # [batch, seq, vocab]

            # Last non-pad token for each sequence.
            last_indices = enc["attention_mask"].sum(dim=1) - 1

            batch_logprobs = []
            for i in range(logits.shape[0]):
                next_logits = logits[i, last_indices[i], :]
                lp = torch.log_softmax(next_logits.float(), dim=-1)

                row_result = {}
                for letter in batch_letters[i]:
                    tok_id = self._single_token_id(letter)
                    row_result[letter] = float(lp[tok_id].detach().cpu())

                batch_logprobs.append(row_result)

            results.extend(batch_logprobs)

            del enc, out, logits
            cleanup_memory()

        return results

In [ ]:
# =====================================================================
# ANES 2024 loading + recoding
# =====================================================================
# This cell is robust to:
# - manually downloaded ANES CSV or zip
# - uppercase/lowercase ANES variable suffixes
# - negative ANES missing-data codes
# - NumPy string/NaN dtype issues in np.select

ANES_DIR = RAW_DATA_DIR / "anes2024"
ANES_DIR.mkdir(parents=True, exist_ok=True)


def find_or_download_anes_csv() -> Path:
    """
    Find the ANES 2024 CSV locally. Automatic ANES download often fails with 403,
    so this function prioritizes manually downloaded files.

    Expected best location:
      /content/drive/MyDrive/NLP_project/data/external_validation/anes2024/
        anes_timeseries_2024_csv_20250808.csv
    """
    ANES_DIR.mkdir(parents=True, exist_ok=True)

    preferred = ANES_DIR / "anes_timeseries_2024_csv_20250808.csv"
    if preferred.exists():
        print("Using existing ANES CSV:", preferred)
        return preferred

    # Search recursively in case the zip extracted into a subfolder.
    csv_candidates = sorted(
        list(ANES_DIR.rglob("*anes*2024*csv*.csv")) +
        list(ANES_DIR.rglob("*.csv")) +
        list(ANES_DIR.rglob("*.CSV"))
    )
    csv_candidates = [p for p in csv_candidates if not p.name.startswith("._")]

    if csv_candidates:
        print("Using existing ANES CSV:", csv_candidates[0])
        return csv_candidates[0]

    # If user placed the zip manually, extract it.
    zip_candidates = sorted(list(ANES_DIR.rglob("*.zip")))
    for zp in zip_candidates:
        print("Found ANES zip, extracting:", zp)
        with zipfile.ZipFile(zp, "r") as zf:
            zf.extractall(ANES_DIR)

        csv_candidates = sorted(
            list(ANES_DIR.rglob("*anes*2024*csv*.csv")) +
            list(ANES_DIR.rglob("*.csv")) +
            list(ANES_DIR.rglob("*.CSV"))
        )
        csv_candidates = [p for p in csv_candidates if not p.name.startswith("._")]

        if csv_candidates:
            print("Using extracted ANES CSV:", csv_candidates[0])
            return csv_candidates[0]

    # Last attempt: automatic download. This may fail with 403.
    zip_path = ANES_DIR / "anes_timeseries_2024_csv_20250808.zip"
    try:
        download_file(ANES_CSV_ZIP_URL, zip_path)
        if not zipfile.is_zipfile(zip_path):
            raise ValueError(f"Downloaded file is not a zip file: {zip_path}")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(ANES_DIR)

        csv_candidates = sorted(
            list(ANES_DIR.rglob("*anes*2024*csv*.csv")) +
            list(ANES_DIR.rglob("*.csv")) +
            list(ANES_DIR.rglob("*.CSV"))
        )
        csv_candidates = [p for p in csv_candidates if not p.name.startswith("._")]

        if csv_candidates:
            print("Using downloaded ANES CSV:", csv_candidates[0])
            return csv_candidates[0]
    except Exception as e:
        raise FileNotFoundError(
            "Could not find or download ANES CSV. Put this file here:\n"
            f"  {ANES_DIR / 'anes_timeseries_2024_csv_20250808.csv'}\n\n"
            f"Original automatic-download error: {e}"
        )

    raise FileNotFoundError(
        "Could not find ANES CSV. Put this file here:\n"
        f"  {ANES_DIR / 'anes_timeseries_2024_csv_20250808.csv'}"
    )


def load_anes_raw() -> pd.DataFrame:
    csv_path = find_or_download_anes_csv()
    df = pd.read_csv(csv_path, low_memory=False)
    df.columns = [str(c).upper() for c in df.columns]
    print("ANES raw:", df.shape)
    return df


def first_present(df: pd.DataFrame, candidates: List[str], required: bool = True) -> Optional[str]:
    """
    Case-insensitive column lookup. Returns uppercase column name because
    load_anes_raw() uppercases the dataframe columns.
    """
    cols = set(df.columns)
    for c in candidates:
        cu = str(c).upper()
        if cu in cols:
            return cu
    if required:
        raise KeyError(f"None of these columns found: {candidates}")
    return None


def num(s, missing_negative: bool = True):
    """
    Convert to numeric. ANES public-use missing codes are usually negative values.
    """
    x = pd.to_numeric(s, errors="coerce")
    if missing_negative:
        x = x.mask(x < 0)
    return x


def select_labels(index, condlist, choicelist, default=pd.NA):
    """
    Safe string-label equivalent of np.select.
    Avoids NumPy dtype errors from mixing strings and np.nan.
    """
    out = pd.Series(default, index=index, dtype="object")
    for cond, choice in zip(condlist, choicelist):
        cond = pd.Series(cond, index=index).fillna(False)
        out.loc[cond] = choice
    return out


def anes_age_bin(age):
    if pd.isna(age) or age < 18:
        return pd.NA
    if age <= 24:
        return "18–24"
    if age <= 34:
        return "25–34"
    if age <= 44:
        return "35–44"
    if age <= 54:
        return "45–54"
    if age <= 64:
        return "55–64"
    return "65+"


ANES_STATE_CODE_MAP = {
    1: "Alabama", 2: "Alaska", 4: "Arizona", 5: "Arkansas", 6: "California",
    8: "Colorado", 9: "Connecticut", 10: "Delaware", 11: "District of Columbia",
    12: "Florida", 13: "Georgia", 15: "Hawaii", 16: "Idaho", 17: "Illinois",
    18: "Indiana", 19: "Iowa", 20: "Kansas", 21: "Kentucky", 22: "Louisiana",
    23: "Maine", 24: "Maryland", 25: "Massachusetts", 26: "Michigan",
    27: "Minnesota", 28: "Mississippi", 29: "Missouri", 30: "Montana",
    31: "Nebraska", 32: "Nevada", 33: "New Hampshire", 34: "New Jersey",
    35: "New Mexico", 36: "New York", 37: "North Carolina", 38: "North Dakota",
    39: "Ohio", 40: "Oklahoma", 41: "Oregon", 42: "Pennsylvania",
    44: "Rhode Island", 45: "South Carolina", 46: "South Dakota",
    47: "Tennessee", 48: "Texas", 49: "Utah", 50: "Vermont", 51: "Virginia",
    53: "Washington", 54: "West Virginia", 55: "Wisconsin", 56: "Wyoming",
}


def recode_anes(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)

    case_col = first_present(df, ["V240001"], required=False)
    out["anes_case_id"] = df[case_col] if case_col else np.arange(len(df))

    # Prefer full-sample-with-PAPI weights when available. The ANES codebook recommends
    # pre-election weights for pre-election-only outcomes.
    w_pre = first_present(
        df,
        ["V240107A", "V240108A", "V240105A", "V240103A", "V240102A", "V240101A"],
        required=False,
    )
    w_post = first_present(
        df,
        ["V240107B", "V240108B", "V240105B", "V240103B", "V240102B", "V240101B"],
        required=False,
    )

    out["weight_pre"] = num(df[w_pre]) if w_pre else 1.0
    out["weight_post"] = num(df[w_post]) if w_post else out["weight_pre"]
    out["weight"] = out["weight_pre"].where(out["weight_pre"].gt(0), 1.0)

    # -----------------------------
    # Demographic prompt features
    # -----------------------------

    sex_col = first_present(df, ["V241550"], required=False)
    if sex_col:
        sex = num(df[sex_col])
        out["SEX"] = sex.map({1: "Male", 2: "Female"})

    age_col = first_present(df, ["V241458X", "V241457"], required=False)
    if age_col:
        age = num(df[age_col])
        out["AGE_BINS"] = age.map(anes_age_bin)

    race_col = first_present(df, ["V241501X"], required=False)
    if race_col:
        race = num(df[race_col])
        out["RACE_ETHNICITY"] = race.map({
            1: "White",
            2: "Black",
            3: "Hispanic",
            4: "Asian/PI",
            5: "Indigenous",
            6: "Multiple",
        })

    educ_col = first_present(df, ["V241465X", "V241463"], required=False)
    if educ_col:
        educ = num(df[educ_col])
        if educ_col == "V241465X":
            out["EDUCATION_LEVEL"] = educ.map({
                1: "Less than high school",
                2: "High school",
                3: "Some college",
                4: "Bachelor's degree",
                5: "Graduate degree",
            })
        else:
            out["EDUCATION_LEVEL"] = select_labels(
                df.index,
                [
                    educ.between(1, 8),
                    educ.eq(9),
                    educ.between(10, 12),
                    educ.between(13, 15),
                    educ.ge(16),
                ],
                [
                    "Less than high school",
                    "High school",
                    "Some college",
                    "Bachelor's degree",
                    "Graduate degree",
                ],
            )

    income_col = first_present(df, ["V241567X", "V241566X"], required=False)
    if income_col:
        inc = num(df[income_col])
        if income_col == "V241567X":
            out["INCTOT_BINS"] = inc.map({
                1: "$1–$14,999",
                2: "$15,000–$29,999",
                3: "$30,000–$49,999",
                4: "$50,000–$74,999",
                5: "$75,000–$149,999",
                6: "$150,000+",
            })
        else:
            out["INCTOT_BINS"] = select_labels(
                df.index,
                [
                    inc.between(1, 4),
                    inc.between(5, 10),
                    inc.between(11, 14),
                    inc.between(15, 19),
                    inc.between(20, 25),
                    inc.ge(26),
                ],
                [
                    "$1–$14,999",
                    "$15,000–$29,999",
                    "$30,000–$49,999",
                    "$50,000–$74,999",
                    "$75,000–$149,999",
                    "$150,000+",
                ],
            )

    mar_col = first_present(df, ["V241461X", "V241459"], required=False)
    if mar_col:
        mar = num(df[mar_col])
        out["MARITAL_STATUS"] = mar.map({
            1: "Married",
            2: "Widowed",
            3: "Divorced",
            4: "Separated",
            5: "Never married",
            6: "Never married",
        })

    emp_col = first_present(df, ["V241488X"], required=False)
    if emp_col:
        emp = num(df[emp_col])
        out["EMPLOYMENT_STATUS"] = emp.map({
            1: "Employed",
            2: "Employed",
            3: "Employed",
            4: "Unemployed",
            5: "Not in labor force",
            6: "Not in labor force",
            7: "Not in labor force",
            8: "Not in labor force",
        })

    health_col = first_present(df, ["V241571"], required=False)
    if health_col:
        h = num(df[health_col])
        out["HEALTH_INSURANCE"] = h.map({
            1: "With health insurance coverage",
            2: "No health insurance coverage",
        })

    # State of registration/voting when available.
    st_col = first_present(df, ["V241023", "V242058X", "V241017"], required=False)
    if st_col:
        st = num(df[st_col])
        out["STATE_FIPS_OR_CODE"] = st
        out["STATE_NAME"] = st.map(ANES_STATE_CODE_MAP)

    # -----------------------------
    # Behavioral ground truths
    # -----------------------------

    # Pre-election summary party of presidential vote/intent/preference:
    # 10/20/30 Dem, 11/21/31 Rep, 12/22/32 Other.
    if "V241075X" in df.columns:
        x = num(df["V241075X"])
        out["BEHAV_pres_party_summary"] = select_labels(
            df.index,
            [
                x.isin([10, 20, 30]),
                x.isin([11, 21, 31]),
                x.isin([12, 22, 32]),
            ],
            [
                "Democrat",
                "Republican",
                "Other",
            ],
        )

    # Two-party-or-not-vote item:
    # 1 Harris/Democrat, 2 Trump/Republican, 3 would not vote.
    if "V241049" in df.columns:
        x = num(df["V241049"])
        out["BEHAV_two_party_or_nonvote"] = x.map({
            1: "Democrat",
            2: "Republican",
            3: "Would not vote",
        })

    # Summary party ID:
    # 1 strong Dem, 2 not strong Dem, 3 independent-Dem,
    # 4 independent, 5 independent-Rep, 6 not strong Rep, 7 strong Rep.
    if "V241227X" in df.columns:
        x = num(df["V241227X"])
        out["BEHAV_party_id"] = select_labels(
            df.index,
            [
                x.isin([1, 2, 3]),
                x.eq(4),
                x.isin([5, 6, 7]),
            ],
            [
                "Democrat",
                "Independent",
                "Republican",
            ],
        )
    elif "V241221" in df.columns:
        x = num(df["V241221"])
        out["BEHAV_party_id"] = x.map({
            1: "Democrat",
            2: "Republican",
            3: "Independent",
            0: "Other/no preference",
            5: "Other/no preference",
        })

    # Ideology:
    # 1-3 liberal, 4 moderate, 5-7 conservative, 99 haven't thought much.
    if "V241177" in df.columns:
        x = pd.to_numeric(df["V241177"], errors="coerce")
        out["BEHAV_ideology_3"] = select_labels(
            df.index,
            [
                x.isin([1, 2, 3]),
                x.eq(4),
                x.isin([5, 6, 7]),
                x.eq(99),
            ],
            [
                "Liberal",
                "Moderate",
                "Conservative",
                "Unsure",
            ],
        )

    # Normalize object missing values.
    out = out.replace({None: pd.NA})

    print("ANES recoded columns:", out.columns.tolist())
    behavior_cols = [c for c in out.columns if c.startswith("BEHAV_")]
    if behavior_cols:
        print("Non-missing behavior counts:")
        display(out[behavior_cols].notna().sum().sort_values(ascending=False))

    demo_cols = [c for c in ["SEX", "AGE_BINS", "RACE_ETHNICITY", "EDUCATION_LEVEL", "INCTOT_BINS", "STATE_NAME"] if c in out.columns]
    if demo_cols:
        print("Non-missing demographic counts:")
        display(out[demo_cols].notna().sum().sort_values(ascending=False))

    return out

In [ ]:
# =====================================================================
# Build behavior tasks: slim test-run version
# =====================================================================
# Kept:
#   1. pres_party_summary = generic Democratic / Republican / Other vote preference
#   2. ideology_3 = Liberal / Moderate / Conservative / Unsure
#
# Dropped for this slim test:
#   - two_party_or_nonvote
#   - party_id

BEHAVIOR_TASKS = {
    "pres_party_summary": {
        "target_col": "BEHAV_pres_party_summary",
        "classes": ["Democrat", "Republican", "Other"],
        "question": (
            "In a generic U.S. presidential election between a Democratic candidate "
            "and a Republican candidate, which candidate would this respondent most "
            "likely vote for or prefer?"
        ),
        "weight_col": "weight_pre",
    },

    "ideology_3": {
        "target_col": "BEHAV_ideology_3",
        "classes": ["Liberal", "Moderate", "Conservative", "Unsure"],
        "question": (
            "How would this respondent most likely describe their political ideology?"
        ),
        "weight_col": "weight_pre",
    },
}

print("Behavior tasks:")
for name, task in BEHAVIOR_TASKS.items():
    print(f"- {name}: target={task['target_col']}, classes={task['classes']}")

In [ ]:
# =====================================================================
# ACS/IPUMS loading and minimal wrangling for state aggregation
# =====================================================================
STATEFIP_map = {
    1: "Alabama", 2: "Alaska", 4: "Arizona", 5: "Arkansas", 6: "California",
    8: "Colorado", 9: "Connecticut", 10: "Delaware", 11: "District of Columbia",
    12: "Florida", 13: "Georgia", 15: "Hawaii", 16: "Idaho", 17: "Illinois",
    18: "Indiana", 19: "Iowa", 20: "Kansas", 21: "Kentucky", 22: "Louisiana",
    23: "Maine", 24: "Maryland", 25: "Massachusetts", 26: "Michigan",
    27: "Minnesota", 28: "Mississippi", 29: "Missouri", 30: "Montana",
    31: "Nebraska", 32: "Nevada", 33: "New Hampshire", 34: "New Jersey",
    35: "New Mexico", 36: "New York", 37: "North Carolina", 38: "North Dakota",
    39: "Ohio", 40: "Oklahoma", 41: "Oregon", 42: "Pennsylvania",
    44: "Rhode Island", 45: "South Carolina", 46: "South Dakota",
    47: "Tennessee", 48: "Texas", 49: "Utah", 50: "Vermont", 51: "Virginia",
    53: "Washington", 54: "West Virginia", 55: "Wisconsin", 56: "Wyoming",
}


def wrangle_raw_acs(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.copy()
    df.columns = [str(c).upper() for c in df.columns]

    if "PERWT" in df.columns:
        df["PERWT"] = pd.to_numeric(df["PERWT"], errors="coerce")
    else:
        df["PERWT"] = 1.0

    # Apply a close version of your original filters if raw IPUMS columns are present.
    if "AGE" in df.columns:
        df["AGE"] = pd.to_numeric(df["AGE"], errors="coerce")
        df = df[(df["AGE"] >= 18) & (df["AGE"] <= 80)].copy()

    if "GQ" in df.columns:
        df["GQ"] = pd.to_numeric(df["GQ"], errors="coerce")
        df = df[df["GQ"].isin([1, 2, 5])].copy()

    if "SEX" in df.columns:
        df["SEX"] = pd.to_numeric(df["SEX"], errors="coerce")
        df = df[df["SEX"].isin([1, 2])].copy()
        df["SEX"] = df["SEX"].map({1: "Male", 2: "Female"})

    if "STATEFIP" in df.columns:
        df["STATEFIP"] = pd.to_numeric(df["STATEFIP"], errors="coerce")
        df["STATE_NAME"] = df["STATEFIP"].map(STATEFIP_map)

    if "AGE_BINS" not in df.columns and "AGE" in df.columns:
        age = pd.to_numeric(df["AGE"], errors="coerce")
        df["AGE_BINS"] = select_labels(
            df.index,
            [
                age.between(18, 24),
                age.between(25, 34),
                age.between(35, 44),
                age.between(45, 54),
                age.between(55, 64),
                age.ge(65),
            ],
            [
                "18–24",
                "25–34",
                "35–44",
                "45–54",
                "55–64",
                "65+",
            ],
        )

    if "RACE_ETHNICITY" not in df.columns and {"RACE", "HISPAN"}.issubset(df.columns):
        df["RACE"] = pd.to_numeric(df["RACE"], errors="coerce")
        df["HISPAN"] = pd.to_numeric(df["HISPAN"], errors="coerce")

        race_map = {
            1: "White",
            2: "Black",
            3: "Indigenous",
            4: "Chinese",
            5: "Japanese",
            6: "Other",
            7: "Other",
            8: "Multiple",
            9: "Multiple",
        }

        df["RACE_ETHNICITY"] = pd.Series(pd.NA, index=df.index, dtype="object")
        df.loc[df["HISPAN"].isin([1, 2, 3, 4]), "RACE_ETHNICITY"] = "Hispanic"
        non_hisp = ~df["HISPAN"].isin([1, 2, 3, 4])
        df.loc[non_hisp, "RACE_ETHNICITY"] = df.loc[non_hisp, "RACE"].map(race_map)

    if "EDUCATION_LEVEL" not in df.columns and "EDUC" in df.columns:
        e = pd.to_numeric(df["EDUC"], errors="coerce")
        df["EDUCATION_LEVEL"] = select_labels(
            df.index,
            [
                e.between(0, 5),
                e.eq(6),
                e.isin([7, 8, 9]),
                e.eq(10),
                e.ge(11),
            ],
            [
                "Less than high school",
                "High school",
                "Some college",
                "Bachelor's degree",
                "Graduate degree",
            ],
        )

    if "INCTOT_BINS" not in df.columns and "INCTOT" in df.columns:
        inc = pd.to_numeric(df["INCTOT"], errors="coerce")
        inc = inc.mask(inc < 0)
        df["INCTOT_BINS"] = select_labels(
            df.index,
            [
                inc.between(0, 14999),
                inc.between(15000, 29999),
                inc.between(30000, 49999),
                inc.between(50000, 74999),
                inc.between(75000, 149999),
                inc.ge(150000),
            ],
            [
                "$1–$14,999",
                "$15,000–$29,999",
                "$30,000–$49,999",
                "$50,000–$74,999",
                "$75,000–$149,999",
                "$150,000+",
            ],
        )

    if "HEALTH_INSURANCE" not in df.columns and "HCOVANY" in df.columns:
        h = pd.to_numeric(df["HCOVANY"], errors="coerce")
        df["HEALTH_INSURANCE"] = h.map({
            1: "No health insurance coverage",
            2: "With health insurance coverage",
        })

    if "ENGLISH_STATUS" not in df.columns and "SPEAKENG" in df.columns:
        s = pd.to_numeric(df["SPEAKENG"], errors="coerce")
        df["ENGLISH_STATUS"] = s.map({
            1: "Does not speak English",
            2: "Speaks English",
            3: "Speaks only English",
            4: "Speaks English very well",
            5: "Speaks English well",
            6: "Speaks English, but not well",
        })

    if "MARITAL_STATUS" not in df.columns and "MARST" in df.columns:
        m = pd.to_numeric(df["MARST"], errors="coerce")
        df["MARITAL_STATUS"] = m.map({
            1: "Married, spouse present",
            2: "Married, spouse absent",
            3: "Separated",
            4: "Divorced",
            5: "Widowed",
            6: "Never married/single",
        })

    if "EMPLOYMENT_STATUS" not in df.columns and "EMPSTAT" in df.columns:
        e = pd.to_numeric(df["EMPSTAT"], errors="coerce")
        df["EMPLOYMENT_STATUS"] = e.map({
            1: "Employed",
            2: "Unemployed",
            3: "Not in labor force",
        })

    if "OWNERSHP_STATUS" not in df.columns and "OWNERSHP" in df.columns:
        o = pd.to_numeric(df["OWNERSHP"], errors="coerce")
        df["OWNERSHP_STATUS"] = o.map({
            1: "Owned or being bought",
            2: "Rented",
        })

    if "WORKER_CLASS" not in df.columns and "CLASSWKR" in df.columns:
        c = pd.to_numeric(df["CLASSWKR"], errors="coerce")
        df["WORKER_CLASS"] = c.map({
            1: "Self-employed",
            2: "Works for wages",
        })

    if "TRANSPORT_METHOD" not in df.columns and "TRANWORK" in df.columns:
        t = pd.to_numeric(df["TRANWORK"], errors="coerce")
        df["TRANSPORT_METHOD"] = t.map({
            10: "Auto, truck, or van",
            11: "Auto",
            12: "Driver",
            13: "Passenger",
            14: "Truck",
            15: "Van",
            20: "Motorcycle",
            31: "Bus",
            32: "Bus",
            33: "Bus or streetcar",
            34: "Light rail, streetcar, or trolley",
            35: "Streetcar or trolley",
            36: "Subway or elevated",
            37: "Train",
            38: "Taxicab or ride-hailing",
            39: "Ferryboat",
            50: "Bicycle",
            60: "Walked only",
            70: "Other",
            80: "Worked at home",
        })

    if "NCHILD" in df.columns:
        df["NCHILD"] = pd.to_numeric(df["NCHILD"], errors="coerce")
    if "UHRSWORK" in df.columns:
        df["UHRSWORK"] = pd.to_numeric(df["UHRSWORK"], errors="coerce")

    keep_cols = ["PERWT"] + [c for c in FEATURE_LABELS if c in df.columns]
    df = df[keep_cols].copy()

    required = ["STATE_NAME", "SEX", "AGE_BINS", "RACE_ETHNICITY", "EDUCATION_LEVEL"]
    required = [c for c in required if c in df.columns]
    df = df.dropna(subset=required).copy()

    df["PERWT"] = pd.to_numeric(df["PERWT"], errors="coerce").fillna(1.0)
    df = df[df["PERWT"] > 0].copy()

    print("ACS/IPUMS usable:", df.shape)
    display(df.head())
    return df


def load_acs_clean() -> pd.DataFrame:
    if ACS_CLEAN_PATH.exists():
        print("Loading clean ACS:", ACS_CLEAN_PATH)
        df = pd.read_parquet(ACS_CLEAN_PATH)
        if "PERWT" not in df.columns:
            df["PERWT"] = 1.0
        return df

    for p in RAW_ACS_PATH_CANDIDATES:
        if p.exists():
            print("Loading raw ACS/IPUMS:", p)
            raw = pd.read_csv(p, low_memory=False)
            df = wrangle_raw_acs(raw)
            try:
                df.to_parquet(ACS_CLEAN_PATH, index=False)
                print("Saved clean ACS to:", ACS_CLEAN_PATH)
            except Exception as e:
                print("Could not save clean ACS:", e)
            return df

    raise FileNotFoundError(
        "No ACS clean parquet or raw usa_00009.csv.gz found. "
        "Put usa_00009.csv.gz in DATA_DIR or update RAW_ACS_PATH_CANDIDATES."
    )

In [ ]:
# =====================================================================
# Scoring loops with progress checkpointing
# =====================================================================
def prepare_task_rows(df: pd.DataFrame, task_name: str, task: dict, n: int, seed: int) -> pd.DataFrame:
    target_col = task['target_col']
    weight_col = task.get('weight_col', 'weight')
    need_cols = [target_col, weight_col, 'SEX', 'AGE_BINS', 'RACE_ETHNICITY', 'EDUCATION_LEVEL']
    valid = df.dropna(subset=[c for c in need_cols if c in df.columns]).copy()
    valid = valid[valid[target_col].isin(task['classes'])].copy()
    valid[weight_col] = pd.to_numeric(valid[weight_col], errors='coerce').fillna(1.0)
    valid = valid[valid[weight_col] > 0]
    if len(valid) == 0:
        raise ValueError(f'No valid ANES rows for task {task_name}')
    sample_n = min(n, len(valid))
    sampled = valid.sample(n=sample_n, replace=False, weights=valid[weight_col], random_state=seed).copy()
    sampled['task_name'] = task_name
    sampled['row_id'] = [f'{task_name}_{i}' for i in range(len(sampled))]
    sampled['row_seed'] = seed + np.arange(len(sampled)) * 1009
    sampled['true_label'] = sampled[target_col]
    sampled['eval_weight'] = sampled[weight_col]
    return sampled.reset_index(drop=True)


def plot_progress(progress_df: pd.DataFrame, path: Path, title: str):
    if progress_df.empty:
        return
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for key, g in progress_df.groupby(['model_label','condition','task_name']):
        label = ' / '.join(key)
        ax.plot(g['n_scored'], g['brier'], marker='o', linewidth=1, markersize=3, label=label)
    ax.set_xlabel('Rows scored')
    ax.set_ylabel('Cumulative Brier score')
    ax.set_title(title)
    ax.legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc='upper left')
    fig.tight_layout()
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)


def compute_running_metrics(df: pd.DataFrame, classes: List[str], weight_col='eval_weight') -> dict:
    w = df[weight_col].astype(float).values
    y = df['true_label'].values
    pred = df['pred_label'].values
    correct = (y == pred).astype(float)
    p_true = df['p_true'].astype(float).clip(1e-6, 1.0).values
    brier_rows = []
    for _, row in df.iterrows():
        yy = np.array([1.0 if row['true_label'] == c else 0.0 for c in classes])
        pp = np.array([row[f'prob_{c}'] for c in classes], dtype=float)
        brier_rows.append(np.sum((pp - yy)**2))
    brier_rows = np.array(brier_rows)
    return {
        'n_scored': len(df),
        'accuracy': weighted_mean(correct, w),
        'brier': weighted_mean(brier_rows, w),
        'log_loss': weighted_mean(-np.log(p_true), w),
    }


def score_behavior_rows(
    scorer: VLLMLetterScorer,
    rows: pd.DataFrame,
    task_name: str,
    task: dict,
    model_label: str,
    condition: str,
    out_prefix: Path,
    batch_size: int = BATCH_SIZE,
) -> pd.DataFrame:
    classes = task['classes']
    partial_path = out_prefix.with_suffix('.partial.parquet')
    final_path = out_prefix.with_suffix('.parquet')
    progress_path = OUT_DIR / 'progress' / f'progress_{safe_name(model_label)}_{safe_name(condition)}_{safe_name(task_name)}.csv'
    progress_plot_path = OUT_DIR / 'progress' / f'progress_{safe_name(model_label)}_{safe_name(condition)}_{safe_name(task_name)}.png'

    if final_path.exists():
        print('Loading existing final scores:', final_path)
        return pd.read_parquet(final_path)

    # Resume from partial if available.
    done = pd.DataFrame()
    done_ids = set()
    if partial_path.exists():
        done = pd.read_parquet(partial_path)
        done_ids = set(done['row_id'])
        print(f'Resuming {task_name}/{model_label}/{condition}: {len(done_ids)} rows already scored')

    rows_to_score = rows[~rows['row_id'].isin(done_ids)].copy().reset_index(drop=True)
    scored_chunks = [done] if len(done) else []
    progress_records = []
    if progress_path.exists():
        progress_records = pd.read_csv(progress_path).to_dict('records')

    for start in tqdm(range(0, len(rows_to_score), batch_size), desc=f'{model_label} {condition} {task_name}'):
        end = min(start + batch_size, len(rows_to_score))
        batch = rows_to_score.iloc[start:end].copy()

        prompts = []
        letters_per_prompt = []
        maps = []
        for _, row in batch.iterrows():
            lettered = make_lettered_options(classes, int(row['row_seed']))
            maps.append(dict(lettered))
            prompt = build_behavior_prompt(row, task, condition, lettered)
            prompts.append((BEHAVIOR_SYSTEM_PROMPT, prompt))
            letters_per_prompt.append([lt for lt, _ in lettered])

        letter_lps = scorer.score_letters_batched(prompts, letters_per_prompt)

        out_rows = []
        for row_idx, (_, row) in enumerate(batch.iterrows()):
            letter_to_class = maps[row_idx]
            lp_by_class = {cls: -100.0 for cls in classes}
            for letter, lp in letter_lps[row_idx].items():
                cls = letter_to_class[letter]
                lp_by_class[cls] = lp
            probs = softmax([lp_by_class[c] for c in classes])
            rec = row.to_dict()
            rec.update({
                'model_label': model_label,
                'condition': condition,
                'task_name': task_name,
                'letter_map_json': json.dumps(letter_to_class),
            })
            for c, p in zip(classes, probs):
                rec[f'logprob_{c}'] = lp_by_class[c]
                rec[f'prob_{c}'] = float(p)
            rec['pred_label'] = classes[int(np.argmax(probs))]
            rec['p_true'] = rec.get(f'prob_{row["true_label"]}', np.nan)
            rec['correct'] = int(rec['pred_label'] == row['true_label'])
            out_rows.append(rec)

        scored_chunks.append(pd.DataFrame(out_rows))
        current = pd.concat(scored_chunks, ignore_index=True)
        current.to_parquet(partial_path, index=False)

        metrics = compute_running_metrics(current, classes)
        metrics.update({'model_label': model_label, 'condition': condition, 'task_name': task_name})
        progress_records.append(metrics)
        progress_df = pd.DataFrame(progress_records)
        progress_df.to_csv(progress_path, index=False)

        if ((start // batch_size) + 1) % CHECKPOINT_EVERY_BATCHES == 0 or end == len(rows_to_score):
            plot_progress(progress_df, progress_plot_path, f'Progress: {model_label} / {condition} / {task_name}')

    final = pd.concat(scored_chunks, ignore_index=True)
    final.to_parquet(final_path, index=False)
    print('Saved final:', final_path)
    return final



In [ ]:
# =====================================================================
# Metrics and plotting
# =====================================================================
def summarize_scores(scores: pd.DataFrame, tasks: Dict[str, dict]) -> pd.DataFrame:
    rows = []
    for (model_label, condition, task_name), g in scores.groupby(['model_label','condition','task_name']):
        task = tasks[task_name]
        classes = task['classes']
        m = compute_running_metrics(g, classes)
        m.update({'model_label': model_label, 'condition': condition, 'task_name': task_name, 'n': len(g)})
        rows.append(m)
    return pd.DataFrame(rows).sort_values(['condition','task_name','model_label'])


def make_metric_delta_table(summary: pd.DataFrame) -> pd.DataFrame:
    base = summary[summary['model_label'] == BASE_RUN_NAME]
    ft = summary[summary['model_label'] == FT_RUN_NAME]
    key = ['condition','task_name']
    merged = base.merge(ft, on=key, suffixes=('_base','_ft'))
    for metric in ['accuracy','brier','log_loss']:
        merged[f'{metric}_delta_ft_minus_base'] = merged[f'{metric}_ft'] - merged[f'{metric}_base']
        if metric == 'accuracy':
            merged[f'{metric}_improvement'] = merged[f'{metric}_ft'] - merged[f'{metric}_base']
        else:
            merged[f'{metric}_improvement'] = merged[f'{metric}_base'] - merged[f'{metric}_ft']
    return merged


def plot_metric_summary(summary: pd.DataFrame, delta: pd.DataFrame):
    plot_dir = OUT_DIR / 'plots'
    for metric in ['brier','log_loss','accuracy']:
        fig, ax = plt.subplots(figsize=(10, 5))
        sub = summary[summary['condition'] == MAIN_CONDITION].copy()
        labels = sorted(sub['task_name'].unique())
        x = np.arange(len(labels))
        width = 0.35
        for offset, model_label in [(-width/2, BASE_RUN_NAME), (width/2, FT_RUN_NAME)]:
            vals = [sub[(sub['task_name']==t) & (sub['model_label']==model_label)][metric].mean() for t in labels]
            ax.bar(x + offset, vals, width, label=model_label)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=30, ha='right')
        ax.set_ylabel(metric)
        ax.set_title(f'{metric}: {BASE_RUN_NAME} vs {FT_RUN_NAME} ({MAIN_CONDITION})')
        ax.legend()
        fig.tight_layout()
        fig.savefig(plot_dir / f'anes_{metric}_base_vs_ft.png', bbox_inches='tight')
        plt.show()

    # Improvement deltas: positive means FT improved.
    fig, ax = plt.subplots(figsize=(10, 4.8))
    sub = delta[delta['condition'] == MAIN_CONDITION].copy()
    labels = sub['task_name'].tolist()
    x = np.arange(len(labels))
    width = 0.28
    for i, metric in enumerate(['brier','log_loss','accuracy']):
        vals = sub[f'{metric}_improvement'].values
        ax.bar(x + (i-1)*width, vals, width, label=f'{metric} improvement')
    ax.axhline(0, linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.set_title('FT improvement over base by behavior task (positive = better)')
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT_DIR / 'plots' / 'anes_ft_improvement_deltas.png', bbox_inches='tight')
    plt.show()


def cell_level_metrics(scores: pd.DataFrame, tasks: Dict[str, dict], cell_features=None) -> pd.DataFrame:
    if cell_features is None:
        cell_features = ['SEX','AGE_BINS','RACE_ETHNICITY','EDUCATION_LEVEL']
    records = []
    for (model_label, condition, task_name), g in scores.groupby(['model_label','condition','task_name']):
        classes = tasks[task_name]['classes']
        use_features = [c for c in cell_features if c in g.columns]
        if not use_features:
            continue
        for cell, cg in g.groupby(use_features, dropna=True):
            if not isinstance(cell, tuple):
                cell = (cell,)
            w = cg['eval_weight'].astype(float)
            if w.sum() <= 0 or len(cg) < 5:
                continue
            for c in classes:
                obs = weighted_mean((cg['true_label'] == c).astype(float), w)
                pred = weighted_mean(cg[f'prob_{c}'].astype(float), w)
                rec = {'model_label': model_label, 'condition': condition, 'task_name': task_name, 'class': c, 'obs_share': obs, 'pred_share': pred, 'abs_error': abs(pred-obs), 'cell_n': len(cg), 'cell_weight': w.sum()}
                rec.update({f'cell_{feat}': val for feat, val in zip(use_features, cell)})
                records.append(rec)
    return pd.DataFrame(records)


def plot_cell_mae(cell_metrics: pd.DataFrame):
    if cell_metrics.empty:
        return
    summ = cell_metrics.groupby(['model_label','condition','task_name']).apply(lambda g: weighted_mean(g['abs_error'], g['cell_weight'])).reset_index(name='cell_weighted_mae')
    summ.to_csv(OUT_DIR / 'tables' / 'cell_weighted_mae.csv', index=False)
    sub = summ[summ['condition'] == MAIN_CONDITION]
    fig, ax = plt.subplots(figsize=(10, 5))
    labels = sorted(sub['task_name'].unique())
    x = np.arange(len(labels))
    width = 0.35
    for offset, model_label in [(-width/2, BASE_RUN_NAME), (width/2, FT_RUN_NAME)]:
        vals = [sub[(sub['task_name']==t) & (sub['model_label']==model_label)]['cell_weighted_mae'].mean() for t in labels]
        ax.bar(x+offset, vals, width, label=model_label)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.set_ylabel('Weighted cell-level MAE')
    ax.set_title('Within-demographic-cell behavioral validity')
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT_DIR / 'plots' / 'anes_cell_weighted_mae.png', bbox_inches='tight')
    plt.show()



In [ ]:
# =====================================================================
# FINAL PRE-REAL-RUN PATCH
# Put this after Cell 14 and before Cell 15.
# =====================================================================
# Fixes:
# 1. Correct next-token logits for left-padded Transformers batches.
# 2. Avoid double-weighting after weighted sampling.
# 3. Save prompt text, answer-option maps, class logprobs/probs, and letter logprobs.

# If sampling with weights, use unit eval weights to avoid double-weighting.
ANES_SAMPLE_WITH_WEIGHTS = True
ANES_EVAL_WEIGHT_AFTER_WEIGHTED_SAMPLE = 1.0

ACS_SAMPLE_WITH_WEIGHTS = True
ACS_EVAL_WEIGHT_AFTER_WEIGHTED_SAMPLE = 1.0


@torch.no_grad()
def fixed_score_letters_batched(
    self,
    prompts: List[Tuple[str, str]],
    letters_per_prompt: List[List[str]],
) -> List[Dict[str, float]]:
    """
    Corrected version for left-padded decoder-only models.

    Since tokenizer.padding_side = "left", the final prompt token is at index -1
    for every sequence in the padded batch. So use logits[:, -1, :].
    """
    results = []

    full_prefixes = [
        self._build_prefix(sys_p, user_p) + PRIMING_SUFFIX
        for sys_p, user_p in prompts
    ]

    for start in range(0, len(full_prefixes), HF_SCORE_BATCH_SIZE):
        end = min(start + HF_SCORE_BATCH_SIZE, len(full_prefixes))
        batch_texts = full_prefixes[start:end]
        batch_letters = letters_per_prompt[start:end]

        enc = self.tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_MODEL_LEN,
        )

        first_device = next(self.model.parameters()).device
        enc = {k: v.to(first_device) for k, v in enc.items()}

        out = self.model(**enc)
        logits = out.logits

        batch_logprobs = []
        for i in range(logits.shape[0]):
            next_logits = logits[i, -1, :]
            lp = torch.log_softmax(next_logits.float(), dim=-1)

            row_result = {}
            for letter in batch_letters[i]:
                tok_id = self._single_token_id(letter)
                row_result[letter] = float(lp[tok_id].detach().cpu())

            batch_logprobs.append(row_result)

        results.extend(batch_logprobs)

        del enc, out, logits
        cleanup_memory()

    return results


VLLMLetterScorer.score_letters_batched = fixed_score_letters_batched
print("Patched VLLMLetterScorer.score_letters_batched for left-padding correctness.")


def prepare_task_rows(df: pd.DataFrame, task_name: str, task: dict, n: int, seed: int) -> pd.DataFrame:
    target_col = task["target_col"]
    weight_col = task.get("weight_col", "weight")

    need_cols = [
        target_col,
        weight_col,
        "SEX",
        "AGE_BINS",
        "RACE_ETHNICITY",
        "EDUCATION_LEVEL",
    ]

    valid = df.dropna(subset=[c for c in need_cols if c in df.columns]).copy()
    valid = valid[valid[target_col].isin(task["classes"])].copy()
    valid[weight_col] = pd.to_numeric(valid[weight_col], errors="coerce").fillna(1.0)
    valid = valid[valid[weight_col] > 0].copy()

    if len(valid) == 0:
        raise ValueError(f"No valid ANES rows for task {task_name}")

    sample_n = min(n, len(valid))
    sample_weights = valid[weight_col] if ANES_SAMPLE_WITH_WEIGHTS else None

    sampled = valid.sample(
        n=sample_n,
        replace=False,
        weights=sample_weights,
        random_state=seed,
    ).copy()

    sampled["task_name"] = task_name
    sampled["row_id"] = [f"{task_name}_{i}" for i in range(len(sampled))]
    sampled["row_seed"] = seed + np.arange(len(sampled)) * 1009
    sampled["true_label"] = sampled[target_col]

    sampled["survey_weight"] = sampled[weight_col].astype(float)

    # Avoid double-weighting after weighted sampling.
    if ANES_SAMPLE_WITH_WEIGHTS:
        sampled["eval_weight"] = ANES_EVAL_WEIGHT_AFTER_WEIGHTED_SAMPLE
    else:
        sampled["eval_weight"] = sampled["survey_weight"]

    sampled["sampled_with_survey_weights"] = ANES_SAMPLE_WITH_WEIGHTS

    print(f"{task_name}: eligible={len(valid)}, sampled={len(sampled)}, weighted_sample={ANES_SAMPLE_WITH_WEIGHTS}")
    print("Sampled true-label counts:", sampled["true_label"].value_counts(dropna=False).to_dict())

    return sampled.reset_index(drop=True)


def sample_acs_by_state(acs: pd.DataFrame, n_per_state: int, seed: int) -> pd.DataFrame:
    chunks = []
    rng = np.random.default_rng(seed)

    valid = acs.dropna(subset=["STATE_NAME", "PERWT"]).copy()
    valid = valid[valid["PERWT"] > 0].copy()

    for state, sdf in valid.groupby("STATE_NAME"):
        if state == "Puerto Rico" or pd.isna(state):
            continue

        n = min(n_per_state, len(sdf))
        if n == 0:
            continue

        replace = len(sdf) < n_per_state
        rs = int(rng.integers(0, 2**31 - 1))

        sample_weights = sdf["PERWT"] if ACS_SAMPLE_WITH_WEIGHTS else None

        chunk = sdf.sample(
            n=n_per_state if replace else n,
            replace=replace,
            weights=sample_weights,
            random_state=rs,
        ).copy()

        chunk["row_id"] = [f"state_{safe_name(state)}_{i}" for i in range(len(chunk))]
        chunk["row_seed"] = seed + np.arange(len(chunk)) * 9173 + len(chunks) * 100000
        chunk["task_name"] = "state_generic_vote"
        chunk["true_label"] = "Unsure"  # placeholder only
        chunk["survey_weight"] = chunk["PERWT"].astype(float)

        # Avoid double-weighting after weighted ACS sampling.
        if ACS_SAMPLE_WITH_WEIGHTS:
            chunk["eval_weight"] = ACS_EVAL_WEIGHT_AFTER_WEIGHTED_SAMPLE
        else:
            chunk["eval_weight"] = chunk["survey_weight"]

        chunk["sampled_with_perwt"] = ACS_SAMPLE_WITH_WEIGHTS
        chunks.append(chunk)

    out = pd.concat(chunks, ignore_index=True)
    print("ACS state simulation rows:", out.shape)
    print("ACS sampled with PERWT:", ACS_SAMPLE_WITH_WEIGHTS)
    return out


def score_behavior_rows(
    scorer: VLLMLetterScorer,
    rows: pd.DataFrame,
    task_name: str,
    task: dict,
    model_label: str,
    condition: str,
    out_prefix: Path,
    batch_size: int = BATCH_SIZE,
) -> pd.DataFrame:
    classes = task["classes"]

    partial_path = out_prefix.with_suffix(".partial.parquet")
    final_path = out_prefix.with_suffix(".parquet")

    progress_path = OUT_DIR / "progress" / f"progress_{safe_name(model_label)}_{safe_name(condition)}_{safe_name(task_name)}.csv"
    progress_plot_path = OUT_DIR / "progress" / f"progress_{safe_name(model_label)}_{safe_name(condition)}_{safe_name(task_name)}.png"

    if final_path.exists():
        print("Loading existing final scores:", final_path)
        return pd.read_parquet(final_path)

    done = pd.DataFrame()
    done_ids = set()

    if partial_path.exists():
        done = pd.read_parquet(partial_path)
        done_ids = set(done["row_id"])
        print(f"Resuming {task_name}/{model_label}/{condition}: {len(done_ids)} rows already scored")

    rows_to_score = rows[~rows["row_id"].isin(done_ids)].copy().reset_index(drop=True)
    scored_chunks = [done] if len(done) else []

    progress_records = []
    if progress_path.exists():
        progress_records = pd.read_csv(progress_path).to_dict("records")

    for start in tqdm(
        range(0, len(rows_to_score), batch_size),
        desc=f"{model_label} {condition} {task_name}",
    ):
        end = min(start + batch_size, len(rows_to_score))
        batch = rows_to_score.iloc[start:end].copy()

        prompts = []
        letters_per_prompt = []
        maps = []
        prompt_texts = []
        option_jsons = []

        for _, row in batch.iterrows():
            lettered = make_lettered_options(classes, int(row["row_seed"]))
            letter_map = dict(lettered)

            prompt = build_behavior_prompt(row, task, condition, lettered)

            maps.append(letter_map)
            prompt_texts.append(prompt)
            option_jsons.append(json.dumps([{"letter": lt, "class": cls} for lt, cls in lettered]))

            prompts.append((BEHAVIOR_SYSTEM_PROMPT, prompt))
            letters_per_prompt.append([lt for lt, _ in lettered])

        letter_lps = scorer.score_letters_batched(prompts, letters_per_prompt)

        out_rows = []

        for row_idx, (_, row) in enumerate(batch.iterrows()):
            letter_to_class = maps[row_idx]
            lp_by_class = {cls: -100.0 for cls in classes}

            for letter, lp in letter_lps[row_idx].items():
                cls = letter_to_class[letter]
                lp_by_class[cls] = lp

            probs = softmax([lp_by_class[c] for c in classes])

            rec = row.to_dict()
            rec.update({
                "model_label": model_label,
                "model_name_or_path": scorer.model_name,
                "condition": condition,
                "task_name": task_name,
                "classes_json": json.dumps(classes),
                "letter_map_json": json.dumps(letter_to_class),
                "letter_logprob_json": json.dumps(letter_lps[row_idx]),
                "answer_options_json": option_jsons[row_idx],
                "system_prompt": BEHAVIOR_SYSTEM_PROMPT,
                "user_prompt": prompt_texts[row_idx],
            })

            for c, p in zip(classes, probs):
                rec[f"logprob_{c}"] = lp_by_class[c]
                rec[f"prob_{c}"] = float(p)

            rec["pred_label"] = classes[int(np.argmax(probs))]
            rec["p_true"] = rec.get(f"prob_{row['true_label']}", np.nan)
            rec["correct"] = int(rec["pred_label"] == row["true_label"])

            out_rows.append(rec)

        scored_chunks.append(pd.DataFrame(out_rows))

        current = pd.concat(scored_chunks, ignore_index=True)
        current.to_parquet(partial_path, index=False)

        metrics = compute_running_metrics(current, classes)
        metrics.update({
            "model_label": model_label,
            "condition": condition,
            "task_name": task_name,
        })

        progress_records.append(metrics)
        progress_df = pd.DataFrame(progress_records)
        progress_df.to_csv(progress_path, index=False)

        if ((start // batch_size) + 1) % CHECKPOINT_EVERY_BATCHES == 0 or end == len(rows_to_score):
            plot_progress(
                progress_df,
                progress_plot_path,
                f"Progress: {model_label} / {condition} / {task_name}",
            )

    final = pd.concat(scored_chunks, ignore_index=True)
    final.to_parquet(final_path, index=False)

    print("Saved final:", final_path)
    return final


print("Patched prepare_task_rows, sample_acs_by_state, and score_behavior_rows.")

In [ ]:
# =====================================================================
# State simulation helpers
# =====================================================================
STATE_VOTE_TASK = {
    'target_col': 'STATE_SIM_GENERIC_VOTE',
    'classes': ['Democrat', 'Republican', 'Other', 'Would not vote', 'Unsure'],
    'question': 'In a generic U.S. presidential election between a Democratic candidate and a Republican candidate, which option would this respondent most likely choose?',
    'weight_col': 'PERWT',
}


def sample_acs_by_state(acs: pd.DataFrame, n_per_state: int, seed: int) -> pd.DataFrame:
    chunks = []
    rng = np.random.default_rng(seed)
    valid = acs.dropna(subset=['STATE_NAME','PERWT']).copy()
    valid = valid[valid['PERWT'] > 0]
    for state, sdf in valid.groupby('STATE_NAME'):
        if state == 'Puerto Rico' or pd.isna(state):
            continue
        n = min(n_per_state, len(sdf))
        if n == 0:
            continue
        replace = len(sdf) < n_per_state
        rs = int(rng.integers(0, 2**31-1))
        chunk = sdf.sample(n=n_per_state if replace else n, replace=replace, weights=sdf['PERWT'], random_state=rs).copy()
        chunk['row_id'] = [f'state_{safe_name(state)}_{i}' for i in range(len(chunk))]
        chunk['row_seed'] = seed + np.arange(len(chunk)) * 9173 + len(chunks) * 100000
        chunk['task_name'] = 'state_generic_vote'
        chunk['true_label'] = 'Unsure'  # placeholder; not used for state aggregation
        chunk['eval_weight'] = chunk['PERWT']
        chunks.append(chunk)
    out = pd.concat(chunks, ignore_index=True)
    print('ACS state simulation rows:', out.shape)
    return out


def load_medsl_state_returns() -> pd.DataFrame:
    csv_path = RAW_DATA_DIR / 'medsl_2024_president_state.csv'
    download_file(MEDSL_STATE_RESULTS_URL, csv_path)
    returns = pd.read_csv(csv_path)
    returns.columns = [c.lower() for c in returns.columns]
    returns['party_simplified'] = returns['party_simplified'].astype(str).str.upper()
    returns['votes'] = pd.to_numeric(returns['votes'], errors='coerce')
    # Use only non-write-in row votes, but keep all DEMOCRAT/REPUBLICAN rows; mode TOTAL where present.
    if 'mode' in returns.columns:
        mask_total = returns['mode'].isna() | returns['mode'].astype(str).str.upper().eq('TOTAL')
        tmp = returns[mask_total].copy()
        if len(tmp) >= 50:
            returns = tmp
    by = returns[returns['party_simplified'].isin(['DEMOCRAT','REPUBLICAN'])].groupby(['state','party_simplified'])['votes'].sum().unstack(fill_value=0).reset_index()
    by['official_dem_two_party'] = by['DEMOCRAT'] / (by['DEMOCRAT'] + by['REPUBLICAN'])
    by['state_name_title'] = by['state'].str.title()
    by.loc[by['state'].eq('DISTRICT OF COLUMBIA'), 'state_name_title'] = 'District of Columbia'
    return by


def aggregate_state_predictions(scores: pd.DataFrame, returns: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (model_label, condition, state), g in scores.groupby(['model_label','condition','STATE_NAME']):
        w = g['eval_weight'].astype(float)
        p_dem = weighted_mean(g['prob_Democrat'], w)
        p_rep = weighted_mean(g['prob_Republican'], w)
        p_other = weighted_mean(g.get('prob_Other', pd.Series(np.nan, index=g.index)), w)
        denom = p_dem + p_rep
        pred_two_party = p_dem / denom if denom > 0 else np.nan
        rows.append({'model_label': model_label, 'condition': condition, 'STATE_NAME': state, 'pred_dem': p_dem, 'pred_rep': p_rep, 'pred_other': p_other, 'pred_dem_two_party': pred_two_party, 'n': len(g), 'weight_sum': w.sum()})
    pred = pd.DataFrame(rows)
    out = pred.merge(returns[['state_name_title','official_dem_two_party','DEMOCRAT','REPUBLICAN']], left_on='STATE_NAME', right_on='state_name_title', how='left')
    out['state_abs_error'] = (out['pred_dem_two_party'] - out['official_dem_two_party']).abs()
    return out


def plot_state_results(state_eval: pd.DataFrame):
    if state_eval.empty:
        return
    state_eval.to_csv(OUT_DIR / 'tables' / 'state_vote_share_predictions.csv', index=False)
    sub = state_eval[(state_eval['condition'] == MAIN_CONDITION) & state_eval['official_dem_two_party'].notna()].copy()

    fig, ax = plt.subplots(figsize=(6, 6))
    for model_label, g in sub.groupby('model_label'):
        ax.scatter(g['official_dem_two_party'], g['pred_dem_two_party'], label=model_label, alpha=0.8)
    ax.plot([0,1],[0,1], linestyle='--', linewidth=1)
    ax.set_xlabel('Official 2024 Democratic two-party share')
    ax.set_ylabel('Model predicted Democratic two-party share')
    ax.set_title('State-level election simulation validity')
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT_DIR / 'plots' / 'state_vote_share_scatter.png', bbox_inches='tight')
    plt.show()

    mae = sub.groupby('model_label').apply(lambda g: weighted_mean(g['state_abs_error'], g['DEMOCRAT'] + g['REPUBLICAN'])).reset_index(name='turnout_weighted_state_mae')
    mae.to_csv(OUT_DIR / 'tables' / 'state_vote_share_mae.csv', index=False)
    display(mae)

    # Top state errors.
    fig, ax = plt.subplots(figsize=(10, 5))
    top = sub.sort_values('state_abs_error', ascending=False).groupby('model_label').head(10)
    labels = top['STATE_NAME'] + ' / ' + top['model_label']
    ax.barh(labels, top['state_abs_error'])
    ax.invert_yaxis()
    ax.set_xlabel('Absolute error')
    ax.set_title('Largest state-level absolute errors')
    fig.tight_layout()
    fig.savefig(OUT_DIR / 'plots' / 'state_vote_share_top_errors.png', bbox_inches='tight')
    plt.show()



In [ ]:
# =====================================================================
# Run ANES microdata experiment
# =====================================================================
all_scores = []

if RUN_ANES_MICRODATA:
    anes_raw = load_anes_raw()
    anes = recode_anes(anes_raw)
    anes.to_parquet(OUT_DIR / 'tables' / 'anes_2024_recoded_for_behavioral_spillover.parquet', index=False)

    task_rows = {}
    for i, (task_name, task) in enumerate(BEHAVIOR_TASKS.items()):
        try:
            tr = prepare_task_rows(anes, task_name, task, n=N_ANES_PER_TASK, seed=RANDOM_SEED + i*10000)
            task_rows[task_name] = tr
            print(task_name, tr.shape, tr['true_label'].value_counts(dropna=False).to_dict())
        except Exception as e:
            print('Skipping task due to error:', task_name, e)

    for model_label, model_path in [(BASE_RUN_NAME, BASE_MODEL_PATH), (FT_RUN_NAME, FT_MODEL_PATH)]:
        print('\n' + '='*80)
        print('Loading model:', model_label, model_path)
        print('='*80)
        scorer = VLLMLetterScorer(model_path)
        try:
            for condition in PROMPT_CONDITIONS:
                for task_name, rows in task_rows.items():
                    task = BEHAVIOR_TASKS[task_name]
                    out_prefix = OUT_DIR / 'scores' / f'anes_{safe_name(model_label)}_{safe_name(condition)}_{safe_name(task_name)}'
                    scored = score_behavior_rows(scorer, rows, task_name, task, model_label, condition, out_prefix)
                    all_scores.append(scored)
        finally:
            scorer.close()

    if all_scores:
        anes_scores = pd.concat(all_scores, ignore_index=True)
        anes_scores.to_parquet(OUT_DIR / 'scores' / 'anes_all_behavior_scores.parquet', index=False)
        summary = summarize_scores(anes_scores, BEHAVIOR_TASKS)
        summary.to_csv(OUT_DIR / 'tables' / 'anes_behavior_metrics_summary.csv', index=False)
        delta = make_metric_delta_table(summary)
        delta.to_csv(OUT_DIR / 'tables' / 'anes_behavior_metric_deltas.csv', index=False)
        display(summary)
        display(delta)
        plot_metric_summary(summary, delta)
        cell_metrics = cell_level_metrics(anes_scores, BEHAVIOR_TASKS)
        cell_metrics.to_csv(OUT_DIR / 'tables' / 'anes_cell_level_predictions.csv', index=False)
        plot_cell_mae(cell_metrics)



In [ ]:
# =====================================================================
# Analyze and save response-probability distributions / mode-collapse diagnostics
# =====================================================================
# This cell uses the already-saved logprob_* and prob_* columns.
# It creates long-format CSVs and plots so you can inspect whether fine-tuning
# changes response distributions or reduces mode collapse.

def scores_to_long_probs(scores: pd.DataFrame, tasks: Dict[str, dict]) -> pd.DataFrame:
    records = []

    for task_name, task in tasks.items():
        classes = task["classes"]
        sub = scores[scores["task_name"] == task_name].copy()

        for _, row in sub.iterrows():
            base_rec = {
                "row_id": row.get("row_id"),
                "anes_case_id": row.get("anes_case_id"),
                "model_label": row.get("model_label"),
                "condition": row.get("condition"),
                "task_name": task_name,
                "true_label": row.get("true_label"),
                "pred_label": row.get("pred_label"),
                "correct": row.get("correct"),
                "eval_weight": row.get("eval_weight", 1.0),

                # Demographic fields for later slicing.
                "SEX": row.get("SEX"),
                "AGE_BINS": row.get("AGE_BINS"),
                "RACE_ETHNICITY": row.get("RACE_ETHNICITY"),
                "EDUCATION_LEVEL": row.get("EDUCATION_LEVEL"),
                "INCTOT_BINS": row.get("INCTOT_BINS"),
                "STATE_NAME": row.get("STATE_NAME"),
            }

            for cls in classes:
                records.append({
                    **base_rec,
                    "response_class": cls,
                    "logprob": row.get(f"logprob_{cls}", np.nan),
                    "prob": row.get(f"prob_{cls}", np.nan),
                    "is_true_class": int(row.get("true_label") == cls),
                    "is_pred_class": int(row.get("pred_label") == cls),
                })

    return pd.DataFrame(records)


def add_distribution_diagnostics(scores: pd.DataFrame, tasks: Dict[str, dict]) -> pd.DataFrame:
    out = scores.copy()

    for task_name, task in tasks.items():
        classes = task["classes"]
        mask = out["task_name"].eq(task_name)
        prob_cols = [f"prob_{c}" for c in classes if f"prob_{c}" in out.columns]

        if not prob_cols:
            continue

        probs = out.loc[mask, prob_cols].astype(float).clip(1e-12, 1.0)
        entropy = -(probs * np.log(probs)).sum(axis=1)
        max_prob = probs.max(axis=1)
        effective_num_classes = np.exp(entropy)

        out.loc[mask, "pred_entropy"] = entropy
        out.loc[mask, "pred_max_prob"] = max_prob
        out.loc[mask, "effective_num_classes"] = effective_num_classes

    return out


def summarize_distribution_diagnostics(scores_diag: pd.DataFrame) -> pd.DataFrame:
    records = []

    for (model_label, condition, task_name), g in scores_diag.groupby(
        ["model_label", "condition", "task_name"]
    ):
        w = pd.to_numeric(g["eval_weight"], errors="coerce").fillna(1.0)

        rec = {
            "model_label": model_label,
            "condition": condition,
            "task_name": task_name,
            "n": len(g),
            "weighted_mean_entropy": weighted_mean(g["pred_entropy"], w),
            "weighted_mean_max_prob": weighted_mean(g["pred_max_prob"], w),
            "weighted_mean_effective_num_classes": weighted_mean(g["effective_num_classes"], w),
        }

        # Predicted label distribution.
        for label, share in g["pred_label"].value_counts(normalize=True).items():
            rec[f"pred_share_{label}"] = share

        # Average probability assigned to each class.
        task = BEHAVIOR_TASKS[task_name]
        for cls in task["classes"]:
            col = f"prob_{cls}"
            if col in g.columns:
                rec[f"mean_prob_{cls}"] = weighted_mean(g[col], w)

        records.append(rec)

    return pd.DataFrame(records).sort_values(["condition", "task_name", "model_label"])


def plot_probability_distributions(long_probs: pd.DataFrame):
    plot_dir = OUT_DIR / "plots"
    plot_dir.mkdir(parents=True, exist_ok=True)

    # Boxplots of probability assigned to each response class.
    for task_name in sorted(long_probs["task_name"].dropna().unique()):
        for condition in sorted(long_probs["condition"].dropna().unique()):
            sub = long_probs[
                (long_probs["task_name"] == task_name)
                & (long_probs["condition"] == condition)
            ].copy()

            if sub.empty:
                continue

            classes = BEHAVIOR_TASKS[task_name]["classes"]
            labels = []
            data = []

            for model_label in [BASE_RUN_NAME, FT_RUN_NAME]:
                for cls in classes:
                    vals = sub[
                        (sub["model_label"] == model_label)
                        & (sub["response_class"] == cls)
                    ]["prob"].dropna().astype(float).values

                    if len(vals):
                        labels.append(f"{model_label}\n{cls}")
                        data.append(vals)

            if not data:
                continue

            fig, ax = plt.subplots(figsize=(max(10, len(labels) * 0.9), 5))
            ax.boxplot(data, labels=labels, showfliers=False)
            ax.set_ylabel("Predicted probability assigned to response")
            ax.set_title(f"Response probability distributions: {task_name} / {condition}")
            ax.tick_params(axis="x", rotation=45)
            fig.tight_layout()

            path = plot_dir / f"prob_distribution_{safe_name(task_name)}_{safe_name(condition)}.png"
            fig.savefig(path, bbox_inches="tight")
            plt.show()

    # Entropy / max-prob diagnostics.
    diag_path = OUT_DIR / "tables" / "anes_scores_with_distribution_diagnostics.parquet"
    scores_diag = pd.read_parquet(diag_path)

    for metric in ["pred_entropy", "pred_max_prob", "effective_num_classes"]:
        fig, ax = plt.subplots(figsize=(10, 5))

        groups = []
        labels = []

        for condition in sorted(scores_diag["condition"].dropna().unique()):
            for task_name in sorted(scores_diag["task_name"].dropna().unique()):
                for model_label in [BASE_RUN_NAME, FT_RUN_NAME]:
                    vals = scores_diag[
                        (scores_diag["condition"] == condition)
                        & (scores_diag["task_name"] == task_name)
                        & (scores_diag["model_label"] == model_label)
                    ][metric].dropna().astype(float).values

                    if len(vals):
                        groups.append(vals)
                        labels.append(f"{condition}\n{task_name}\n{model_label}")

        if groups:
            ax.boxplot(groups, labels=labels, showfliers=False)
            ax.set_ylabel(metric)
            ax.set_title(f"Mode-collapse diagnostic: {metric}")
            ax.tick_params(axis="x", rotation=60)
            fig.tight_layout()
            fig.savefig(plot_dir / f"mode_collapse_{metric}.png", bbox_inches="tight")
            plt.show()


# Load ANES scores if not already in memory.
anes_scores_path = OUT_DIR / "scores" / "anes_all_behavior_scores.parquet"

if not anes_scores_path.exists():
    raise FileNotFoundError(
        f"Could not find ANES scores at:\n{anes_scores_path}\n\n"
        "Run the ANES scoring cell first."
    )

anes_scores_loaded = pd.read_parquet(anes_scores_path)
anes_scores_diag = add_distribution_diagnostics(anes_scores_loaded, BEHAVIOR_TASKS)

# Save wide diagnostic table.
diag_path = OUT_DIR / "tables" / "anes_scores_with_distribution_diagnostics.parquet"
anes_scores_diag.to_parquet(diag_path, index=False)

# Save long-format logprob/prob table.
long_probs = scores_to_long_probs(anes_scores_diag, BEHAVIOR_TASKS)
long_probs_path_csv = OUT_DIR / "tables" / "anes_response_logprobs_long.csv"
long_probs_path_parquet = OUT_DIR / "tables" / "anes_response_logprobs_long.parquet"

long_probs.to_csv(long_probs_path_csv, index=False)
long_probs.to_parquet(long_probs_path_parquet, index=False)

# Save summary diagnostics.
dist_summary = summarize_distribution_diagnostics(anes_scores_diag)
dist_summary_path = OUT_DIR / "tables" / "anes_distribution_mode_collapse_summary.csv"
dist_summary.to_csv(dist_summary_path, index=False)

display(dist_summary)

plot_probability_distributions(long_probs)

print("Saved:")
print(" ", diag_path)
print(" ", long_probs_path_csv)
print(" ", long_probs_path_parquet)
print(" ", dist_summary_path)

In [ ]:
# =====================================================================
# Unweighted bar-chart response distributions
# =====================================================================
# This cell intentionally plots WITHOUT survey/eval weights.
#
# Why:
# - The ANES rows may already have been sampled using survey weights.
# - For visual mode-collapse diagnostics, raw sampled shares are easier to interpret.
# - If 5 out of 50 sampled respondents are Unsure, the plot should show 0.10.
#
# It creates:
#   1. True sampled response distribution
#   2. Model top-predicted response distribution
#   3. Model mean probability assigned to each response
#
# All are unweighted.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


def load_anes_scores_for_distribution() -> pd.DataFrame:
    all_path = OUT_DIR / "scores" / "anes_all_behavior_scores.parquet"

    if all_path.exists():
        print("Loading:", all_path)
        return pd.read_parquet(all_path)

    score_files = sorted((OUT_DIR / "scores").glob("anes_*.parquet"))
    score_files = [
        p for p in score_files
        if "partial" not in p.name and "all_behavior_scores" not in p.name
    ]

    if not score_files:
        raise FileNotFoundError(
            f"No ANES score files found in:\n{OUT_DIR / 'scores'}\n\n"
            "Run the ANES scoring cell first."
        )

    print("Concatenating individual score files:")
    for p in score_files:
        print(" ", p.name)

    return pd.concat([pd.read_parquet(p) for p in score_files], ignore_index=True)


def unweighted_class_distribution(df, label_col, classes):
    """
    Return unweighted share of each class in label_col.
    """
    out = []
    denom = len(df)

    for cls in classes:
        share = float(df[label_col].eq(cls).sum() / denom) if denom > 0 else np.nan
        out.append({
            "response_class": cls,
            "share": share,
            "count": int(df[label_col].eq(cls).sum()),
            "n": int(denom),
        })

    return pd.DataFrame(out)


def unweighted_mean_probability_distribution(df, classes):
    """
    Return unweighted mean predicted probability assigned to each class.
    """
    out = []
    denom = len(df)

    for cls in classes:
        col = f"prob_{cls}"
        if col not in df.columns:
            mean_prob = np.nan
        else:
            mean_prob = float(pd.to_numeric(df[col], errors="coerce").mean())

        out.append({
            "response_class": cls,
            "share": mean_prob,
            "count": np.nan,
            "n": int(denom),
        })

    return pd.DataFrame(out)


def make_grouped_bar(
    plot_df,
    title,
    ylabel,
    save_path,
    group_col="series",
    class_col="response_class",
    value_col="share",
):
    classes = list(plot_df[class_col].dropna().unique())
    series = list(plot_df[group_col].dropna().unique())

    x = np.arange(len(classes))
    width = 0.8 / max(len(series), 1)

    fig, ax = plt.subplots(figsize=(max(8, len(classes) * 1.6), 5))

    for i, s in enumerate(series):
        vals = []
        sub = plot_df[plot_df[group_col] == s]

        for cls in classes:
            v = sub.loc[sub[class_col] == cls, value_col]
            vals.append(float(v.iloc[0]) if len(v) else np.nan)

        ax.bar(
            x + (i - (len(series) - 1) / 2) * width,
            vals,
            width,
            label=s,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(classes, rotation=25, ha="right")
    ax.set_ylim(0, max(1.0, np.nanmax(plot_df[value_col].values) * 1.15))
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()

    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, bbox_inches="tight")
    plt.show()

    print("Saved:", save_path)


anes_scores = load_anes_scores_for_distribution()

plot_dir = OUT_DIR / "plots"
table_dir = OUT_DIR / "tables"
plot_dir.mkdir(parents=True, exist_ok=True)
table_dir.mkdir(parents=True, exist_ok=True)

bar_records = []

for task_name, task in BEHAVIOR_TASKS.items():
    classes = task["classes"]

    for condition in PROMPT_CONDITIONS:
        sub = anes_scores[
            (anes_scores["task_name"] == task_name)
            & (anes_scores["condition"] == condition)
        ].copy()

        if sub.empty:
            print(f"Skipping empty subset: {task_name} / {condition}")
            continue

        # True ANES labels are duplicated across model rows, so drop duplicates.
        if "anes_case_id" in sub.columns:
            true_sub = sub.drop_duplicates(["anes_case_id", "task_name", "condition"])
        else:
            true_sub = sub.drop_duplicates(["row_id", "task_name", "condition"])

        true_dist = unweighted_class_distribution(
            true_sub,
            label_col="true_label",
            classes=classes,
        )
        true_dist["series"] = "True sampled ANES"

        pred_parts = [true_dist]
        meanprob_parts = []

        for model_label in [BASE_RUN_NAME, FT_RUN_NAME]:
            m = sub[sub["model_label"] == model_label].copy()

            if m.empty:
                print(f"Missing model subset: {model_label} / {task_name} / {condition}")
                continue

            pred_dist = unweighted_class_distribution(
                m,
                label_col="pred_label",
                classes=classes,
            )
            pred_dist["series"] = f"{model_label} top prediction"
            pred_parts.append(pred_dist)

            mean_prob_dist = unweighted_mean_probability_distribution(
                m,
                classes=classes,
            )
            mean_prob_dist["series"] = f"{model_label} mean probability"
            meanprob_parts.append(mean_prob_dist)

        pred_plot_df = pd.concat(pred_parts, ignore_index=True)
        pred_plot_df["task_name"] = task_name
        pred_plot_df["condition"] = condition
        pred_plot_df["plot_type"] = "unweighted_top_prediction_distribution"

        meanprob_plot_df = pd.concat(meanprob_parts, ignore_index=True)
        meanprob_plot_df["task_name"] = task_name
        meanprob_plot_df["condition"] = condition
        meanprob_plot_df["plot_type"] = "unweighted_mean_probability_distribution"

        bar_records.append(pred_plot_df)
        bar_records.append(meanprob_plot_df)

        make_grouped_bar(
            pred_plot_df,
            title=f"Unweighted true vs top-predicted response distribution\n{task_name} / {condition}",
            ylabel="Share of sampled respondents",
            save_path=plot_dir / f"unweighted_bar_top_pred_distribution_{safe_name(task_name)}_{safe_name(condition)}.png",
        )

        make_grouped_bar(
            meanprob_plot_df,
            title=f"Unweighted mean predicted probability by response class\n{task_name} / {condition}",
            ylabel="Mean predicted probability",
            save_path=plot_dir / f"unweighted_bar_mean_prob_distribution_{safe_name(task_name)}_{safe_name(condition)}.png",
        )

bar_summary = pd.concat(bar_records, ignore_index=True)
bar_summary_path = table_dir / "anes_unweighted_bar_distribution_summary.csv"
bar_summary.to_csv(bar_summary_path, index=False)

display(bar_summary)

print("Saved summary table:", bar_summary_path)

In [ ]:
# =====================================================================
# Run ACS-to-state election aggregation experiment
# =====================================================================
if RUN_STATE_SIMULATION:
    acs = load_acs_clean()
    acs_rows = sample_acs_by_state(acs, N_ACS_PER_STATE, RANDOM_SEED)
    acs_rows.to_parquet(OUT_DIR / 'tables' / 'acs_state_sim_sample.parquet', index=False)
    state_scores_all = []

    for model_label, model_path in [(BASE_RUN_NAME, BASE_MODEL_PATH), (FT_RUN_NAME, FT_MODEL_PATH)]:
        print('\n' + '='*80)
        print('Loading model for state simulation:', model_label, model_path)
        print('='*80)
        scorer = VLLMLetterScorer(model_path)
        try:
            for condition in PROMPT_CONDITIONS:
                out_prefix = OUT_DIR / 'scores' / f'state_sim_{safe_name(model_label)}_{safe_name(condition)}'
                scored = score_behavior_rows(scorer, acs_rows, 'state_generic_vote', STATE_VOTE_TASK, model_label, condition, out_prefix)
                state_scores_all.append(scored)
        finally:
            scorer.close()

    state_scores = pd.concat(state_scores_all, ignore_index=True)
    state_scores.to_parquet(OUT_DIR / 'scores' / 'state_sim_all_scores.parquet', index=False)
    returns = load_medsl_state_returns()
    returns.to_csv(OUT_DIR / 'tables' / 'medsl_2024_state_two_party_returns.csv', index=False)
    state_eval = aggregate_state_predictions(state_scores, returns)
    plot_state_results(state_eval)



In [ ]:
# =====================================================================
# Final manifest of outputs
# =====================================================================
manifest = []
for p in sorted(OUT_DIR.rglob('*')):
    if p.is_file():
        manifest.append({'path': str(p), 'size_mb': p.stat().st_size / 1e6})
manifest_df = pd.DataFrame(manifest)
manifest_df.to_csv(OUT_DIR / 'output_manifest.csv', index=False)
display(manifest_df.tail(50))
print('Done. Output directory:', OUT_DIR)


Final visualizations

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("/content/drive/MyDrive/NLP_project/data/behavioral_spillover")

RUN_PREFIX = "qwen25_14b_base__vs__qwen25_14b_demog_lora_step07__seed42__"

candidates = sorted([
    p for p in ROOT.glob(f"{RUN_PREFIX}*")
    if p.is_dir() and "anes_real_n1200" in p.name
])

print("Candidate n=1200 runs:")
for p in candidates:
    print(" ", p.name)

if not candidates:
    raise FileNotFoundError(
        f"No n=1200 run found under {ROOT}. "
        "Check that the real-run notebook finished and saved outputs."
    )

OUT_DIR = candidates[-1]

print("\nUsing OUT_DIR:")
print(OUT_DIR)

config_path = OUT_DIR / "run_config.json"
scores_path = OUT_DIR / "scores" / "anes_all_behavior_scores.parquet"
delta_path = OUT_DIR / "tables" / "anes_behavior_metric_deltas.csv"

if config_path.exists():
    config = json.loads(config_path.read_text())
    print("\nrun_config N_ANES_PER_TASK:", config.get("N_ANES_PER_TASK"))

anes_scores = pd.read_parquet(scores_path)
delta = pd.read_csv(delta_path)

print("\nLoaded scores shape:", anes_scores.shape)

check = (
    anes_scores
    .groupby(["model_label", "condition", "task_name"])
    .size()
    .reset_index(name="n_rows")
    .sort_values(["condition", "task_name", "model_label"])
)

display(check)

if not (check["n_rows"] == 1200).all():
    raise ValueError(
        "This is not the real n=1200 run. At least one model/condition/task group "
        "does not have 1200 rows."
    )

print("\nConfirmed: using real n=1200 run.")

In [ ]:
task_map = {
    "ideology_3": "Ideology",
    "pres_party_summary": "Pres. party",
}
condition_map = {
    "full": "Full",
    "no_state": "No state",
}

delta["condition_order"] = delta["condition"].map({"full": 0, "no_state": 1}).fillna(99)
delta["task_order"] = delta["task_name"].map({"ideology_3": 0, "pres_party_summary": 1}).fillna(99)
delta = delta.sort_values(["condition_order", "task_order"]).reset_index(drop=True)

delta["plot_label"] = (
    delta["task_name"].map(task_map).fillna(delta["task_name"])
    + "\n"
    + delta["condition"].map(condition_map).fillna(delta["condition"])
)

fig, axes = plt.subplots(1, 2, figsize=(8.8, 3.4))

for ax, col, title in [
    (axes[0], "brier_improvement", "Brier improvement"),
    (axes[1], "log_loss_improvement", "Log-loss improvement"),
]:
    x = np.arange(len(delta))
    ax.bar(x, delta[col].values)
    ax.axhline(0, linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(delta["plot_label"], rotation=0)
    ax.set_title(title)
    ax.set_ylabel("Base minus fine-tuned\npositive = better")
    ax.grid(axis="y", alpha=0.25)

fig.suptitle("Behavioral spillover from demographic fine-tuning", y=1.03)
fig.tight_layout()

plot_dir = OUT_DIR / "plots"
plot_dir.mkdir(parents=True, exist_ok=True)

out_pdf = plot_dir / "behavioral_spillover_brier_logloss_clean_REAL_N1200.pdf"
out_png = plot_dir / "behavioral_spillover_brier_logloss_clean_REAL_N1200.png"

fig.savefig(out_pdf, bbox_inches="tight")
fig.savefig(out_png, dpi=250, bbox_inches="tight")
plt.show()

print(out_pdf)
print(out_png)

In [ ]:
BASE_RUN_NAME = "qwen25_14b_base"
FT_RUN_NAME = "qwen25_14b_demog_lora_step07"

MODEL_DISPLAY = {
    BASE_RUN_NAME: "Base model",
    FT_RUN_NAME: "Demographic LoRA",
}

TASK_CLASSES = {
    "ideology_3": ["Liberal", "Moderate", "Conservative", "Unsure"],
    "pres_party_summary": ["Democrat", "Republican", "Other"],
}

TASK_DISPLAY = {
    "ideology_3": "Ideology",
    "pres_party_summary": "Presidential party preference",
}

CONDITION_DISPLAY = {
    "full": "Full prompt",
    "no_state": "No-state prompt",
}

def safe_name(x):
    return "".join(ch if ch.isalnum() or ch in "._-" else "_" for ch in str(x))

def make_probability_distribution_table(scores, task_name, condition, classes):
    sub = scores[
        (scores["task_name"] == task_name)
        & (scores["condition"] == condition)
    ].copy()

    if sub.empty:
        raise ValueError(f"No rows for {task_name} / {condition}")

    # True labels are duplicated once per model, so deduplicate.
    true_sub = sub.drop_duplicates(["row_id", "task_name", "condition"])
    assert len(true_sub) == 1200, f"Expected 1200 true rows, got {len(true_sub)}"

    records = []

    for cls in classes:
        count = int(true_sub["true_label"].eq(cls).sum())
        prob = count / len(true_sub)

        records.append({
            "task_name": task_name,
            "condition": condition,
            "response_class": cls,
            "series": "Ground truth (ANES)",
            "probability": prob,
            "percentage": 100 * prob,
            "count": count,
            "n": len(true_sub),
        })

    for model_label in [BASE_RUN_NAME, FT_RUN_NAME]:
        m = sub[sub["model_label"] == model_label].copy()
        assert len(m) == 1200, f"Expected 1200 rows for {model_label}, got {len(m)}"

        for cls in classes:
            col = f"prob_{cls}"
            if col not in m.columns:
                raise ValueError(f"Missing column: {col}")

            prob = pd.to_numeric(m[col], errors="coerce").mean()

            records.append({
                "task_name": task_name,
                "condition": condition,
                "response_class": cls,
                "series": MODEL_DISPLAY[model_label],
                "probability": prob,
                "percentage": 100 * prob,
                "count": np.nan,
                "n": len(m),
            })

    return pd.DataFrame(records)

def distribution_distance_summary(plot_df):
    true = (
        plot_df[plot_df["series"] == "Ground truth (ANES)"]
        .set_index("response_class")["probability"]
    )

    rows = []

    for series in ["Base model", "Demographic LoRA"]:
        pred = (
            plot_df[plot_df["series"] == series]
            .set_index("response_class")["probability"]
        )

        common = true.index.intersection(pred.index)

        rows.append({
            "series": series,
            "total_variation_distance": 0.5 * np.abs(pred.loc[common] - true.loc[common]).sum(),
            "mean_abs_class_prob_error": np.abs(pred.loc[common] - true.loc[common]).mean(),
        })

    return pd.DataFrame(rows)

def plot_probability_distribution(plot_df, save_stem, title):
    classes = list(plot_df["response_class"].dropna().unique())
    series_order = ["Ground truth (ANES)", "Base model", "Demographic LoRA"]

    dist_summary = distribution_distance_summary(plot_df)
    base_tv = dist_summary.loc[
        dist_summary["series"] == "Base model",
        "total_variation_distance",
    ].iloc[0]
    ft_tv = dist_summary.loc[
        dist_summary["series"] == "Demographic LoRA",
        "total_variation_distance",
    ].iloc[0]

    x = np.arange(len(classes))
    width = 0.8 / len(series_order)

    fig, ax = plt.subplots(figsize=(max(7, 1.45 * len(classes)), 4.5))

    for i, series in enumerate(series_order):
        sub = plot_df[plot_df["series"] == series]
        vals = [
            float(sub.loc[sub["response_class"] == cls, "percentage"].iloc[0])
            for cls in classes
        ]

        ax.bar(
            x + (i - (len(series_order) - 1) / 2) * width,
            vals,
            width,
            label=series,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(classes, rotation=20, ha="right")
    ax.set_ylabel("Probability mass (%)")
    ax.set_ylim(0, max(100, plot_df["percentage"].max() * 1.18))
    ax.set_title(
        title
        + f"\nTV distance to ANES: base={base_tv:.3f}, LoRA={ft_tv:.3f}"
    )
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()

    out_png = plot_dir / f"{save_stem}_REAL_N1200.png"
    out_pdf = plot_dir / f"{save_stem}_REAL_N1200.pdf"

    fig.savefig(out_png, dpi=250, bbox_inches="tight")
    fig.savefig(out_pdf, bbox_inches="tight")
    plt.show()

    print("Saved:", out_png)
    print("Saved:", out_pdf)

    return dist_summary

all_prob_tables = []
all_distance_tables = []

for task_name in ["ideology_3", "pres_party_summary"]:
    condition = "no_state"
    classes = TASK_CLASSES[task_name]

    plot_df = make_probability_distribution_table(
        anes_scores,
        task_name=task_name,
        condition=condition,
        classes=classes,
    )

    all_prob_tables.append(plot_df)

    title = (
        f"Ground-truth vs model response distributions\n"
        f"{TASK_DISPLAY[task_name]} / {CONDITION_DISPLAY[condition]}"
    )

    save_stem = (
        f"anes_groundtruth_vs_model_probability_distribution_"
        f"{safe_name(task_name)}_{safe_name(condition)}"
    )

    dist_summary = plot_probability_distribution(plot_df, save_stem, title)
    dist_summary["task_name"] = task_name
    dist_summary["condition"] = condition
    all_distance_tables.append(dist_summary)

prob_summary = pd.concat(all_prob_tables, ignore_index=True)
distance_summary = pd.concat(all_distance_tables, ignore_index=True)

prob_summary_path = OUT_DIR / "tables" / "anes_groundtruth_vs_model_probability_distribution_REAL_N1200.csv"
distance_summary_path = OUT_DIR / "tables" / "anes_groundtruth_vs_model_distribution_distance_REAL_N1200.csv"

prob_summary.to_csv(prob_summary_path, index=False)
distance_summary.to_csv(distance_summary_path, index=False)

display(prob_summary)
display(distance_summary)

print("Saved:", prob_summary_path)
print("Saved:", distance_summary_path)

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# 1. Load verified real n=1200 run
# -----------------------------
ROOT = Path("/content/drive/MyDrive/NLP_project/data/behavioral_spillover")
RUN_PREFIX = "qwen25_14b_base__vs__qwen25_14b_demog_lora_step07__seed42__"

candidates = sorted([
    p for p in ROOT.glob(f"{RUN_PREFIX}*")
    if p.is_dir() and "anes_real_n1200" in p.name
])

print("Candidate n=1200 runs:")
for p in candidates:
    print(" ", p.name)

if not candidates:
    raise FileNotFoundError(
        f"No n=1200 run found under {ROOT}. "
        "Check that the real behavioral spillover notebook finished and saved outputs."
    )

OUT_DIR = candidates[-1]
print("\nUsing OUT_DIR:")
print(OUT_DIR)

config_path = OUT_DIR / "run_config.json"
scores_path = OUT_DIR / "scores" / "anes_all_behavior_scores.parquet"
delta_path = OUT_DIR / "tables" / "anes_behavior_metric_deltas.csv"

if config_path.exists():
    config = json.loads(config_path.read_text())
    print("\nrun_config N_ANES_PER_TASK:", config.get("N_ANES_PER_TASK"))

anes_scores = pd.read_parquet(scores_path)
delta = pd.read_csv(delta_path)

# Verify this is the real run.
check = (
    anes_scores
    .groupby(["model_label", "condition", "task_name"])
    .size()
    .reset_index(name="n_rows")
    .sort_values(["condition", "task_name", "model_label"])
)

print("\nScore table verification:")
display(check)

if not (check["n_rows"] == 1200).all():
    raise ValueError(
        "This is not the real n=1200 run. At least one model/condition/task group "
        "does not have 1200 rows."
    )

print("\nConfirmed: using real n=1200 run.")

# -----------------------------
# 2. Clean labels and ordering
# -----------------------------
task_map = {
    "ideology_3": "Ideology",
    "pres_party_summary": "Pres. party",
}

condition_map = {
    "full": "Full",
    "no_state": "No state",
}

delta["condition_order"] = delta["condition"].map({"full": 0, "no_state": 1}).fillna(99)
delta["task_order"] = delta["task_name"].map({"ideology_3": 0, "pres_party_summary": 1}).fillna(99)
delta = delta.sort_values(["condition_order", "task_order"]).reset_index(drop=True)

delta["plot_label"] = (
    delta["task_name"].map(task_map).fillna(delta["task_name"])
    + "\n"
    + delta["condition"].map(condition_map).fillna(delta["condition"])
)

# Compute improvement columns if needed.
if "brier_improvement" not in delta.columns:
    delta["brier_improvement"] = delta["brier_base"] - delta["brier_ft"]

if "log_loss_improvement" not in delta.columns:
    delta["log_loss_improvement"] = delta["log_loss_base"] - delta["log_loss_ft"]

# -----------------------------
# 3. Print tables for copy/paste
# -----------------------------
print("\n=== Raw metrics table for writeup ===")
raw_print_cols = [
    "task_name", "condition",
    "n_base", "n_ft",
    "accuracy_base", "accuracy_ft",
    "brier_base", "brier_ft",
    "log_loss_base", "log_loss_ft",
]
raw_print_cols = [c for c in raw_print_cols if c in delta.columns]

raw_table = delta[raw_print_cols].copy()
for c in raw_table.select_dtypes(include="number").columns:
    raw_table[c] = raw_table[c].round(4)

display(raw_table)
print(raw_table.to_string(index=False))

print("\n=== Improvement table for writeup ===")
improvement_print_cols = [
    "task_name", "condition",
    "accuracy_delta_ft_minus_base",
    "brier_improvement",
    "log_loss_improvement",
]
improvement_print_cols = [c for c in improvement_print_cols if c in delta.columns]

improvement_table = delta[improvement_print_cols].copy()
for c in improvement_table.select_dtypes(include="number").columns:
    improvement_table[c] = improvement_table[c].round(4)

display(improvement_table)
print(improvement_table.to_string(index=False))

# -----------------------------
# 4. Plot raw Brier and log-loss values
# -----------------------------
plot_dir = OUT_DIR / "plots"
plot_dir.mkdir(parents=True, exist_ok=True)

x = np.arange(len(delta))
width = 0.36

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))

# Raw Brier
axes[0].bar(x - width / 2, delta["brier_base"], width, label="Base model")
axes[0].bar(x + width / 2, delta["brier_ft"], width, label="Demographic LoRA")
axes[0].set_title("Raw Brier score")
axes[0].set_ylabel("Brier score lower is better")
axes[0].set_xticks(x)
axes[0].set_xticklabels(delta["plot_label"], rotation=0)
axes[0].grid(axis="y", alpha=0.25)
axes[0].legend()

# Raw log loss
axes[1].bar(x - width / 2, delta["log_loss_base"], width, label="Base model")
axes[1].bar(x + width / 2, delta["log_loss_ft"], width, label="Demographic LoRA")
axes[1].set_title("Raw log loss")
axes[1].set_ylabel("Log loss lower is better")
axes[1].set_xticks(x)
axes[1].set_xticklabels(delta["plot_label"], rotation=0)
axes[1].grid(axis="y", alpha=0.25)
axes[1].legend()

fig.suptitle("ANES behavioral prediction: raw probability metrics", y=1.04)
fig.tight_layout()

raw_pdf = plot_dir / "anes_raw_brier_logloss_REAL_N1200.pdf"
raw_png = plot_dir / "anes_raw_brier_logloss_REAL_N1200.png"

fig.savefig(raw_pdf, bbox_inches="tight")
fig.savefig(raw_png, dpi=250, bbox_inches="tight")
plt.show()

print("\nSaved raw metric plot:")
print(raw_pdf)
print(raw_png)

# -----------------------------
# 5. Plot improvement values
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))

for ax, col, title in [
    (axes[0], "brier_improvement", "Brier improvement"),
    (axes[1], "log_loss_improvement", "Log-loss improvement"),
]:
    ax.bar(x, delta[col].values)
    ax.axhline(0, linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(delta["plot_label"], rotation=0)
    ax.set_title(title)
    ax.set_ylabel("Base minus LoRA\npositive = LoRA better")
    ax.grid(axis="y", alpha=0.25)

fig.suptitle("Behavioral spillover from demographic fine-tuning", y=1.04)
fig.tight_layout()

improve_pdf = plot_dir / "behavioral_spillover_brier_logloss_improvement_REAL_N1200.pdf"
improve_png = plot_dir / "behavioral_spillover_brier_logloss_improvement_REAL_N1200.png"

fig.savefig(improve_pdf, bbox_inches="tight")
fig.savefig(improve_png, dpi=250, bbox_inches="tight")
plt.show()

print("\nSaved improvement plot:")
print(improve_pdf)
print(improve_png)

Final final visualizations

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# 1. Load verified real n=1200 run
# -----------------------------
ROOT = Path("/content/drive/MyDrive/NLP_project/data/behavioral_spillover")
RUN_PREFIX = "qwen25_14b_base__vs__qwen25_14b_demog_lora_step07__seed42__"

candidates = sorted([
    p for p in ROOT.glob(f"{RUN_PREFIX}*")
    if p.is_dir() and "anes_real_n1200" in p.name
])

print("Candidate n=1200 runs:")
for p in candidates:
    print(" ", p.name)

if not candidates:
    raise FileNotFoundError(f"No n=1200 run found under {ROOT}")

OUT_DIR = candidates[-1]
print("\nUsing OUT_DIR:")
print(OUT_DIR)

config_path = OUT_DIR / "run_config.json"
scores_path = OUT_DIR / "scores" / "anes_all_behavior_scores.parquet"
delta_path = OUT_DIR / "tables" / "anes_behavior_metric_deltas.csv"

if config_path.exists():
    config = json.loads(config_path.read_text())
    print("\nrun_config N_ANES_PER_TASK:", config.get("N_ANES_PER_TASK"))

anes_scores = pd.read_parquet(scores_path)
delta = pd.read_csv(delta_path)

# Verify real n=1200 run.
check = (
    anes_scores
    .groupby(["model_label", "condition", "task_name"])
    .size()
    .reset_index(name="n_rows")
    .sort_values(["condition", "task_name", "model_label"])
)

print("\nScore table verification:")
display(check)

if not (check["n_rows"] == 1200).all():
    raise ValueError("This is not the real n=1200 run.")

print("\nConfirmed: using real n=1200 run.")

plot_dir = OUT_DIR / "plots"
table_dir = OUT_DIR / "tables"
plot_dir.mkdir(parents=True, exist_ok=True)
table_dir.mkdir(parents=True, exist_ok=True)

# -----------------------------
# 2. Full-profile metrics only
# -----------------------------
FULL_CONDITION = "full"

task_map = {
    "ideology_3": "Ideology",
    "pres_party_summary": "Pres. party",
}

task_order = {
    "ideology_3": 0,
    "pres_party_summary": 1,
}

delta_full = delta[delta["condition"] == FULL_CONDITION].copy()
delta_full["task_order"] = delta_full["task_name"].map(task_order).fillna(99)
delta_full = delta_full.sort_values("task_order").reset_index(drop=True)

delta_full["plot_label"] = delta_full["task_name"].map(task_map).fillna(delta_full["task_name"])

if "brier_improvement" not in delta_full.columns:
    delta_full["brier_improvement"] = delta_full["brier_base"] - delta_full["brier_ft"]

if "log_loss_improvement" not in delta_full.columns:
    delta_full["log_loss_improvement"] = delta_full["log_loss_base"] - delta_full["log_loss_ft"]

print("\n=== Full-profile raw metrics table ===")
raw_cols = [
    "task_name", "condition",
    "n_base", "n_ft",
    "accuracy_base", "accuracy_ft",
    "brier_base", "brier_ft",
    "log_loss_base", "log_loss_ft",
]
raw_cols = [c for c in raw_cols if c in delta_full.columns]
raw_table = delta_full[raw_cols].copy()

for c in raw_table.select_dtypes(include="number").columns:
    raw_table[c] = raw_table[c].round(4)

display(raw_table)
print(raw_table.to_string(index=False))

print("\n=== Full-profile improvement table ===")
improvement_cols = [
    "task_name", "condition",
    "accuracy_delta_ft_minus_base",
    "brier_improvement",
    "log_loss_improvement",
]
improvement_cols = [c for c in improvement_cols if c in delta_full.columns]
improvement_table = delta_full[improvement_cols].copy()

for c in improvement_table.select_dtypes(include="number").columns:
    improvement_table[c] = improvement_table[c].round(4)

display(improvement_table)
print(improvement_table.to_string(index=False))

# -----------------------------
# 3. Plot full-profile Brier/log-loss improvement
# -----------------------------
x = np.arange(len(delta_full))

fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.6))

for ax, col, title in [
    (axes[0], "brier_improvement", "Brier improvement"),
    (axes[1], "log_loss_improvement", "Log-loss improvement"),
]:
    ax.bar(x, delta_full[col].values)
    ax.axhline(0, linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(delta_full["plot_label"], rotation=0)
    ax.set_title(title)
    ax.set_ylabel("Base minus LoRA\npositive = LoRA better")
    ax.grid(axis="y", alpha=0.25)

fig.suptitle("Behavioral spillover under full-profile prompting", y=1.04)
fig.tight_layout()

improve_pdf = plot_dir / "behavioral_spillover_brier_logloss_improvement_full_profile_REAL_N1200.pdf"
improve_png = plot_dir / "behavioral_spillover_brier_logloss_improvement_full_profile_REAL_N1200.png"

fig.savefig(improve_pdf, bbox_inches="tight")
fig.savefig(improve_png, dpi=250, bbox_inches="tight")
plt.show()

print("\nSaved full-profile improvement plot:")
print(improve_pdf)
print(improve_png)

# -----------------------------
# 4. Full-profile response-distribution plots
# -----------------------------
BASE_RUN_NAME = "qwen25_14b_base"
FT_RUN_NAME = "qwen25_14b_demog_lora_step07"

MODEL_DISPLAY = {
    BASE_RUN_NAME: "Base model",
    FT_RUN_NAME: "Demographic LoRA",
}

TASK_CLASSES = {
    "ideology_3": ["Liberal", "Moderate", "Conservative", "Unsure"],
    "pres_party_summary": ["Democrat", "Republican", "Other"],
}

TASK_DISPLAY = {
    "ideology_3": "Ideology",
    "pres_party_summary": "Presidential party preference",
}

def make_probability_distribution_table(scores, task_name, condition, classes):
    sub = scores[
        (scores["task_name"] == task_name)
        & (scores["condition"] == condition)
    ].copy()

    if sub.empty:
        raise ValueError(f"No rows for {task_name} / {condition}")

    # True labels are duplicated once per model, so deduplicate.
    if "row_id" in sub.columns:
        true_sub = sub.drop_duplicates(["row_id", "task_name", "condition"])
    else:
        non_model_cols = [
            c for c in sub.columns
            if c != "model_label" and not c.startswith("prob_")
        ]
        true_sub = sub.drop_duplicates(non_model_cols)

    assert len(true_sub) == 1200, f"Expected 1200 true rows, got {len(true_sub)}"

    records = []

    for cls in classes:
        count = int(true_sub["true_label"].eq(cls).sum())
        prob = count / len(true_sub)

        records.append({
            "task_name": task_name,
            "condition": condition,
            "response_class": cls,
            "series": "Ground truth (ANES)",
            "probability": prob,
            "percentage": 100 * prob,
            "count": count,
            "n": len(true_sub),
        })

    for model_label in [BASE_RUN_NAME, FT_RUN_NAME]:
        m = sub[sub["model_label"] == model_label].copy()
        assert len(m) == 1200, f"Expected 1200 rows for {model_label}, got {len(m)}"

        for cls in classes:
            col = f"prob_{cls}"
            if col not in m.columns:
                raise ValueError(f"Missing probability column: {col}")

            prob = pd.to_numeric(m[col], errors="coerce").mean()

            records.append({
                "task_name": task_name,
                "condition": condition,
                "response_class": cls,
                "series": MODEL_DISPLAY[model_label],
                "probability": prob,
                "percentage": 100 * prob,
                "count": np.nan,
                "n": len(m),
            })

    return pd.DataFrame(records)

def distribution_distance_summary(plot_df):
    true = (
        plot_df[plot_df["series"] == "Ground truth (ANES)"]
        .set_index("response_class")["probability"]
    )

    rows = []

    for series in ["Base model", "Demographic LoRA"]:
        pred = (
            plot_df[plot_df["series"] == series]
            .set_index("response_class")["probability"]
        )

        common = true.index.intersection(pred.index)

        rows.append({
            "series": series,
            "total_variation_distance": 0.5 * np.abs(pred.loc[common] - true.loc[common]).sum(),
            "mean_abs_class_prob_error": np.abs(pred.loc[common] - true.loc[common]).mean(),
        })

    return pd.DataFrame(rows)

def plot_probability_distribution(plot_df, save_stem, title):
    classes = list(plot_df["response_class"].dropna().unique())
    series_order = ["Ground truth (ANES)", "Base model", "Demographic LoRA"]

    dist_summary = distribution_distance_summary(plot_df)
    base_tv = dist_summary.loc[
        dist_summary["series"] == "Base model",
        "total_variation_distance",
    ].iloc[0]
    ft_tv = dist_summary.loc[
        dist_summary["series"] == "Demographic LoRA",
        "total_variation_distance",
    ].iloc[0]

    x = np.arange(len(classes))
    width = 0.8 / len(series_order)

    fig, ax = plt.subplots(figsize=(max(7, 1.45 * len(classes)), 4.4))

    for i, series in enumerate(series_order):
        sub = plot_df[plot_df["series"] == series]
        vals = [
            float(sub.loc[sub["response_class"] == cls, "percentage"].iloc[0])
            for cls in classes
        ]

        ax.bar(
            x + (i - (len(series_order) - 1) / 2) * width,
            vals,
            width,
            label=series,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(classes, rotation=20, ha="right")
    ax.set_ylabel("Probability mass (%)")
    ax.set_ylim(0, max(100, plot_df["percentage"].max() * 1.18))
    ax.set_title(
        title
        + f"\nTV distance to ANES: base={base_tv:.3f}, LoRA={ft_tv:.3f}"
    )
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()

    out_png = plot_dir / f"{save_stem}.png"
    out_pdf = plot_dir / f"{save_stem}.pdf"

    fig.savefig(out_png, dpi=250, bbox_inches="tight")
    fig.savefig(out_pdf, bbox_inches="tight")
    plt.show()

    print("Saved:", out_png)
    print("Saved:", out_pdf)

    return dist_summary

all_prob_tables = []
all_distance_tables = []

for task_name in ["ideology_3", "pres_party_summary"]:
    classes = TASK_CLASSES[task_name]

    plot_df = make_probability_distribution_table(
        anes_scores,
        task_name=task_name,
        condition=FULL_CONDITION,
        classes=classes,
    )

    all_prob_tables.append(plot_df)

    title = f"Full-profile {TASK_DISPLAY[task_name]} distribution: ANES vs. model predictions"

    save_stem = f"anes_full_profile_distribution_{task_name}_REAL_N1200"

    dist_summary = plot_probability_distribution(plot_df, save_stem, title)
    dist_summary["task_name"] = task_name
    dist_summary["condition"] = FULL_CONDITION
    all_distance_tables.append(dist_summary)

prob_summary = pd.concat(all_prob_tables, ignore_index=True)
distance_summary = pd.concat(all_distance_tables, ignore_index=True)

prob_summary_path = table_dir / "anes_full_profile_probability_distribution_REAL_N1200.csv"
distance_summary_path = table_dir / "anes_full_profile_distribution_distance_REAL_N1200.csv"

prob_summary.to_csv(prob_summary_path, index=False)
distance_summary.to_csv(distance_summary_path, index=False)

print("\n=== Full-profile response distribution table ===")
display(prob_summary)
print(prob_summary.round(4).to_string(index=False))

print("\n=== Full-profile response distribution distance table ===")
display(distance_summary)
print(distance_summary.round(4).to_string(index=False))

print("\nSaved tables:")
print(prob_summary_path)
print(distance_summary_path)

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# 1. Load verified real n=1200 / n=150-per-state run
# -----------------------------
ROOT = Path("/content/drive/MyDrive/NLP_project/data/behavioral_spillover")
RUN_PREFIX = "qwen25_14b_base__vs__qwen25_14b_demog_lora_step07__seed42__"

candidates = sorted([
    p for p in ROOT.glob(f"{RUN_PREFIX}*")
    if p.is_dir()
    and "anes_real_n1200" in p.name
    and "state_real_n150_per_state" in p.name
])

print("Candidate real runs:")
for p in candidates:
    print(" ", p.name)

if not candidates:
    raise FileNotFoundError(f"No verified real run found under {ROOT}")

OUT_DIR = candidates[-1]
print("\nUsing OUT_DIR:")
print(OUT_DIR)

plot_dir = OUT_DIR / "plots"
table_dir = OUT_DIR / "tables"
plot_dir.mkdir(parents=True, exist_ok=True)
table_dir.mkdir(parents=True, exist_ok=True)

config_path = OUT_DIR / "run_config.json"
if config_path.exists():
    config = json.loads(config_path.read_text())
    print("\nrun_config:")
    print("  N_ANES_PER_TASK:", config.get("N_ANES_PER_TASK"))
    print("  N_ACS_PER_STATE:", config.get("N_ACS_PER_STATE"))
    print("  PROMPT_CONDITIONS:", config.get("PROMPT_CONDITIONS"))
    print("  MAIN_CONDITION:", config.get("MAIN_CONDITION"))

# -----------------------------
# 2. Load state-level prediction outputs
# -----------------------------
state_eval_path = table_dir / "state_vote_share_predictions.csv"
mae_path = table_dir / "state_vote_share_mae.csv"

if not state_eval_path.exists():
    raise FileNotFoundError(
        f"Missing {state_eval_path}. "
        "Run the ACS-to-state election aggregation cell in the behavioral notebook first."
    )

state_eval = pd.read_csv(state_eval_path)

print("\nLoaded state_eval:", state_eval.shape)
print("Columns:")
print(list(state_eval.columns))

# Use full-profile condition only.
FULL_CONDITION = "full"
sub = state_eval[
    (state_eval["condition"] == FULL_CONDITION)
    & state_eval["official_dem_two_party"].notna()
].copy()

if sub.empty:
    raise ValueError("No full-profile rows with official election results found.")

# Friendly model labels.
MODEL_NAME = {
    "qwen25_14b_base": "Base model",
    "qwen25_14b_demog_lora_step07": "Demographic LoRA",
}
sub["model_display"] = sub["model_label"].map(MODEL_NAME).fillna(sub["model_label"])

# Basic derived columns.
sub["turnout_two_party"] = sub["DEMOCRAT"] + sub["REPUBLICAN"]
sub["signed_error"] = sub["pred_dem_two_party"] - sub["official_dem_two_party"]
sub["abs_error"] = sub["signed_error"].abs()

# Verify n per state if present.
if "n" in sub.columns:
    n_check = sub.groupby("model_display")["n"].describe()
    print("\nACS profiles per state verification:")
    display(n_check)

# -----------------------------
# 3. Print main MAE table
# -----------------------------
def weighted_mean(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    mask = np.isfinite(x) & np.isfinite(w) & (w > 0)
    if mask.sum() == 0:
        return np.nan
    return float(np.sum(x[mask] * w[mask]) / np.sum(w[mask]))

mae = (
    sub.groupby(["model_label", "model_display"])
    .apply(lambda g: weighted_mean(g["abs_error"], g["turnout_two_party"]))
    .reset_index(name="turnout_weighted_state_mae")
)

mae["turnout_weighted_state_mae"] = mae["turnout_weighted_state_mae"].round(4)
mae_out = table_dir / "state_vote_share_mae_full_profile_REAL_N1200.csv"
mae.to_csv(mae_out, index=False)

print("\n=== Full-profile ACS election MAE table ===")
display(mae)
print(mae.to_string(index=False))
print("Saved:", mae_out)

# -----------------------------
# 4. Print state-level predictions table
# -----------------------------
state_print_cols = [
    "STATE_NAME",
    "model_display",
    "official_dem_two_party",
    "pred_dem_two_party",
    "signed_error",
    "abs_error",
    "pred_dem",
    "pred_rep",
    "pred_other",
    "n",
    "weight_sum",
]

state_print_cols = [c for c in state_print_cols if c in sub.columns]

state_table = sub[state_print_cols].copy()
for c in state_table.select_dtypes(include="number").columns:
    state_table[c] = state_table[c].round(4)

state_table_out = table_dir / "state_vote_share_predictions_full_profile_REAL_N1200.csv"
state_table.to_csv(state_table_out, index=False)

print("\n=== Full-profile state-level prediction table ===")
display(state_table.sort_values(["STATE_NAME", "model_display"]))
print(state_table.sort_values(["STATE_NAME", "model_display"]).to_string(index=False))
print("Saved:", state_table_out)

# -----------------------------
# 5. Print top errors per model
# -----------------------------
top_errors = (
    sub.sort_values(["model_display", "abs_error"], ascending=[True, False])
    .groupby("model_display")
    .head(10)
    [state_print_cols]
    .copy()
)

for c in top_errors.select_dtypes(include="number").columns:
    top_errors[c] = top_errors[c].round(4)

top_errors_out = table_dir / "state_vote_share_top_errors_full_profile_REAL_N1200.csv"
top_errors.to_csv(top_errors_out, index=False)

print("\n=== Top 10 absolute state errors per model ===")
display(top_errors)
print(top_errors.to_string(index=False))
print("Saved:", top_errors_out)

# -----------------------------
# 6. Print per-state improvement table
# -----------------------------
wide = sub.pivot_table(
    index="STATE_NAME",
    columns="model_display",
    values=["pred_dem_two_party", "signed_error", "abs_error"],
    aggfunc="first",
)

wide.columns = [f"{a}_{b}".replace(" ", "_").lower() for a, b in wide.columns]
wide = wide.reset_index()

# Add official share once.
official = (
    sub[["STATE_NAME", "official_dem_two_party", "turnout_two_party"]]
    .drop_duplicates("STATE_NAME")
)
wide = wide.merge(official, on="STATE_NAME", how="left")

if "abs_error_base_model" in wide.columns and "abs_error_demographic_lora" in wide.columns:
    wide["abs_error_change_lora_minus_base"] = (
        wide["abs_error_demographic_lora"] - wide["abs_error_base_model"]
    )
    wide["lora_improves_abs_error"] = wide["abs_error_change_lora_minus_base"] < 0

wide_out = table_dir / "state_vote_share_lora_vs_base_by_state_full_profile_REAL_N1200.csv"
wide_round = wide.copy()
for c in wide_round.select_dtypes(include="number").columns:
    wide_round[c] = wide_round[c].round(4)

wide_round.to_csv(wide_out, index=False)

print("\n=== Per-state LoRA vs base comparison ===")
display(wide_round.sort_values("abs_error_change_lora_minus_base" if "abs_error_change_lora_minus_base" in wide_round.columns else "STATE_NAME"))
print(wide_round.to_string(index=False))
print("Saved:", wide_out)

if "lora_improves_abs_error" in wide.columns:
    n_improved = int(wide["lora_improves_abs_error"].sum())
    n_states = int(wide["lora_improves_abs_error"].notna().sum())
    print(f"\nLoRA improves absolute error in {n_improved}/{n_states} states.")

# -----------------------------
# 7. Plot 1: scatter official vs predicted
# -----------------------------
fig, ax = plt.subplots(figsize=(6.2, 6.0))

for model_display, g in sub.groupby("model_display"):
    ax.scatter(
        g["official_dem_two_party"],
        g["pred_dem_two_party"],
        label=model_display,
        alpha=0.8,
    )

ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
ax.set_xlabel("Official 2024 Democratic two-party share")
ax.set_ylabel("Predicted Democratic two-party share")
ax.set_title("ACS-to-election aggregation under full-profile prompting")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()

scatter_pdf = plot_dir / "acs_election_state_vote_share_scatter_full_profile_REAL_N1200.pdf"
scatter_png = plot_dir / "acs_election_state_vote_share_scatter_full_profile_REAL_N1200.png"

fig.savefig(scatter_pdf, bbox_inches="tight")
fig.savefig(scatter_png, dpi=250, bbox_inches="tight")
plt.show()

print("\nSaved scatter plot:")
print(scatter_pdf)
print(scatter_png)

# -----------------------------
# 8. Plot 2: turnout-weighted MAE bar chart
# -----------------------------
fig, ax = plt.subplots(figsize=(5.5, 3.8))

mae_plot = mae.sort_values("model_display")
ax.bar(mae_plot["model_display"], mae_plot["turnout_weighted_state_mae"])
ax.set_ylabel("Turnout-weighted state MAE")
ax.set_title("State-level election prediction error")
ax.grid(axis="y", alpha=0.25)

for i, v in enumerate(mae_plot["turnout_weighted_state_mae"]):
    ax.text(i, v, f"{v:.3f}", ha="center", va="bottom")

fig.tight_layout()

mae_pdf = plot_dir / "acs_election_state_mae_full_profile_REAL_N1200.pdf"
mae_png = plot_dir / "acs_election_state_mae_full_profile_REAL_N1200.png"

fig.savefig(mae_pdf, bbox_inches="tight")
fig.savefig(mae_png, dpi=250, bbox_inches="tight")
plt.show()

print("\nSaved MAE plot:")
print(mae_pdf)
print(mae_png)

# -----------------------------
# 9. Plot 3: top absolute errors
# -----------------------------
top_plot = (
    sub.sort_values("abs_error", ascending=False)
    .groupby("model_display")
    .head(10)
    .copy()
)
top_plot["label"] = top_plot["STATE_NAME"] + " / " + top_plot["model_display"]

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.barh(top_plot["label"], top_plot["abs_error"])
ax.invert_yaxis()
ax.set_xlabel("Absolute error in Democratic two-party share")
ax.set_title("Largest state-level absolute errors under full-profile prompting")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()

top_pdf = plot_dir / "acs_election_top_state_errors_full_profile_REAL_N1200.pdf"
top_png = plot_dir / "acs_election_top_state_errors_full_profile_REAL_N1200.png"

fig.savefig(top_pdf, bbox_inches="tight")
fig.savefig(top_png, dpi=250, bbox_inches="tight")
plt.show()

print("\nSaved top-error plot:")
print(top_pdf)
print(top_png)